In [2]:
import re
import os
import json
import pandas as pd

save_type = ['Code', 'Math', 'Reasoning_&_Logic', 'Data_Analysis', 'Creative_Writing', 'Instruction_Following_&_Text_Processing', 'Role_Playing_&_Chat', 'Humanities_Arts_&_Social_Sciences', 'Natural_&_Applied_Sciences', 'Business_&_Finance', 'Daily_Life_&_Health', 'General_Knowledge_QA']

def process_datas(k, values, v_clarity_down, v_cg_down, v_clarity_up=10, v_cg_up=10):
    # 筛选条件
    res = []
    for elem in values:
        # print(elem['score']['Clarity']['Score'])
        if  (elem['score']['Clarity']['Score'] >= v_clarity_down and elem['score']['Clarity']['Score'] <= v_clarity_up and elem['score']['Cognitive_Complexity']['Score'] >= v_cg_down and elem['score']['Cognitive_Complexity']['Score'] <= v_cg_up):
            res.append(elem)

    return res
        
def deal_datas_step2(datas, v_clarity_down=8, v_clarity_up=10, v_cg_down=5, v_cg_up=10):
    total_results = []
    others = []
    for k, v in datas.items():
        if k in save_type:
            # 处理数据，传入阈值
            res = process_datas(k, v, v_clarity_down, v_cg_down, v_clarity_up, v_cg_up)
            total_results.append({k: res})
        else:
            others.extend(v)
    res = process_datas('Other', others, v_clarity_down, v_cg_down)
    total_results.append({'Other': res})
    return total_results

def step2_save_filter_datas_gai(data_type_all, v_clarity_down, v_cg_down, v_clarity_up=10, v_cg_up=10):
    str_name = 'clarity_code_over_{}_to_{}_and_cg_over_{}_to_{}'.format(v_clarity_down, v_clarity_up, v_cg_down, v_cg_up)
    res = deal_datas_step2(data_type_all, v_clarity_down=v_clarity_down, v_cg_down=v_cg_down, v_clarity_up=v_clarity_up, v_cg_up=v_cg_up)
    
    path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2'
    save_path = path + '/' + str_name
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    df_dics = {}
    for elem in res:
        for k, v in elem.items():
            df_dics[k] = len(v)
            # print(k, len(v))
            save_path_ = os.path.join(save_path, "{}.jsonl".format(k))
            save_datas(save_path_, v)

    return df_dics

def save_datas(path, values):
    with open(path, 'w', encoding='utf8') as f_save:
        for value in values:
            f_save.write(json.dumps(value, ensure_ascii=False)+'\n')
    print('数据保存完成:::', path)

# 202w + 50w 垂类数据，从sft 开始训练，观察效果
import json 
from tqdm import  tqdm 

def yield_datas(path):
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            yield line

import re
import os
import json
import pandas as pd
from tqdm import tqdm 

def read_datas(path):
    results = []
    try:
        with open(path, 'r', encoding='utf8') as f:
            results = json.load(f)
    except:
        results = []
        with open(path, 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                results.append(cur_)
    return results



In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('/workspace/zxd/Qwen/Qwen3-4B-Instruct-2507', trust_remote_code=True)

In [2]:
code_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/code/code_1.jsonl'
math_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/math/math_1.jsonl'
general_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/general/general_1.jsonl'
swe_t_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/swe_testgen.jsonl'
if_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/instruction-following.jsonl'
science_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/science.jsonl'
swe_l_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/swe_localization.jsonl'
swe_r_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/swe_repair.jsonl'
tool_call_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/tool_calling.jsonl'
chat_if_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/data/chat_if.jsonl'
struct_output_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/data/structured_outputs.jsonl'

In [5]:
code_datas = read_datas(code_path)

# 代码数据，目测，每条数据 /think 后都是 完整可解决的回答，因此这份代码数据直接阶段使用 
code_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Cascade-SFT-Stage-2-code_1.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in enumerate(code_datas):
        if len(elem['messages']) != 2:
            errors += 1
            continue
        cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-code-1-{}'.format(idx), 'output': elem['messages'][1]['content'].split('</think>')[-1].strip()}
        code_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')

In [14]:
# 指令遵循数据，目测，大部分think 数据都无法直接使用，采用tokenizer 来判断，如果编码后的长度小于 3072 tokens，则保留，剩下的采用 235b-no-thinking 模型来蒸馏得到 结果
if_datas = read_datas(if_path)

In [19]:
if_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Cascade-SFT-Stage-2-if.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(if_datas)):
        if elem['thinking'] == True:
            # 判断长度
            cur_length = tokenizer(elem['messages'][1]['content'], return_tensors='pt')['input_ids'].shape[1]
            if cur_length <= 3072:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-if-{}'.format(idx), 'output': elem['messages'][1]['content'].replace('<think>', '').replace('</think>', '').strip()}
            else:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-if-{}'.format(idx), 'output': 'need_235b_distill'}
                errors += 1
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-if-{}'.format(idx), 'output': elem['messages'][1]['content'].strip()}
        if_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
# errors  5208

146385it [07:48, 312.13it/s]


In [22]:
errors

5208

In [21]:
science_datas = read_datas(science_path)

In [25]:
science_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Cascade-SFT-Stage-2-science.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(science_datas)):
        if elem['thinking'] == True:
            # 判断长度
            cur_length = tokenizer(elem['messages'][1]['content'], return_tensors='pt')['input_ids'].shape[1]
            if cur_length <= 3072:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-science-{}'.format(idx), 'output': elem['messages'][1]['content'].replace('<think>', '').replace('</think>', '').strip()}
            else:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-science-{}'.format(idx), 'output': 'need_235b_distill'}
                errors += 1
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-science-{}'.format(idx), 'output': elem['messages'][1]['content'].strip()}
        science_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
print(errors) # 229506

311316it [3:31:42, 24.51it/s]

229506


In [26]:
swe_l_datas = read_datas(swe_l_path)

In [41]:
len(swe_l_datas), swe_l_datas[0]

(92265,
 {'category': 'SWE Localization',
  'source': 'internlm/SWE-Fixer-Train-110K',
  'messages': [{'role': 'user',
    'content': 'Please look through a given GitHub issue and repository structure and provide a list of files that one would need to edit or look at to solve the issue.\n\n### GitHub Problem Description ###\nside-tag order on new update are ordered randomly?\nCreating a new update shows all available side tags, but they show up ordered randomly?\r\n\r\n![image](https://github.com/fedora-infra/bodhi/assets/773148/7bda4940-0470-484f-b5fc-50a29d45aa48)\r\n\r\nI would expect list to show up ordered by sidetag name or by ID.\n\n\n###\n\n### Repository Structure ###\nbodhi-messages\n    bodhi\n        messages\n            __init__.py\n            utils.py\n            schemas\n                __init__.py\n                update.py\n                compose.py\n                errata.py\n                buildroot_override.py\n                base.py\ndevel\n    register-with-

In [27]:
print(len(swe_l_datas))
swe_l_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Cascade-SFT-Stage-2-swe_l.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(swe_l_datas)):
        if elem['thinking'] == True:
            # 判断长度
            cur_length = tokenizer(elem['messages'][1]['content'], return_tensors='pt')['input_ids'].shape[1]
            if cur_length <= 3072:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-swe_l-{}'.format(idx), 'output': elem['messages'][1]['content'].replace('<think>', '').replace('</think>', '').strip()}
            else:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-swe_l-{}'.format(idx), 'output': 'need_235b_distill'}
                errors += 1
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-swe_l-{}'.format(idx), 'output': elem['messages'][1]['content'].strip()}
        swe_l_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
print(errors) # 92265

92265


In [42]:
swe_r_datas[0]

{'category': 'SWE Repair',
 'source': 'princeton-nlp/SWE-bench',
 'messages': [{'role': 'user',
   'content': 'We are currently solving the following issue within our repository. Here is the issue text:\n--- BEGIN ISSUE ---\npkgconfig add URL parameter\nHello,\r\nI have tried to add URL to pkg.generate in:\r\n``` python\r\npkg = import(\'pkgconfig\')\r\nlibs = ...     # the library/libraries users need to link against\r\nh = [\'.\', ...] # subdirectories of ${prefix}/${includedir} to add to header path\r\npkg.generate(libraries : libs,\r\n             subdirs : h,\r\n             version : \'1.0\',\r\n             name : \'libsimple\',\r\n             filebase : \'simple\',\r\n             description : \'A simple demo library.\')\r\n```\r\nbut there is error, so it seems that isn\'t supported yet ?\r\n\r\nAny help appreciated\n\n--- END ISSUE ---\n\nBelow are some code segments, each from a relevant file. One or more of these files may contain bugs.\n\n--- BEGIN FILE ---\n```\n### mes

In [30]:
swe_r_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Cascade-SFT-Stage-2-swe_r.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(swe_r_datas)):
        if elem['thinking'] == True:
            # 判断长度
            cur_length = tokenizer(elem['messages'][1]['content'], return_tensors='pt')['input_ids'].shape[1]
            if cur_length <= 3072:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-swe_l-{}'.format(idx), 'output': elem['messages'][1]['content'].replace('<think>', '').replace('</think>', '').strip()}
            else:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-swe_l-{}'.format(idx), 'output': 'need_235b_distill'}
                errors += 1
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-swe_l-{}'.format(idx), 'output': elem['messages'][1]['content'].strip()}
        swe_r_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
print(errors) # 44688 总量 86907

86907it [26:26, 54.79it/s]

44688


In [31]:
chat_if_datas = read_datas(chat_if_path) # 是多轮次 if，后续可以直接拿来使用，这次先不用
print(len(chat_if_datas))

426009


In [35]:
chat_if_datas[1]
# 是多伦对话，只取单轮对话,这个感觉不是很好的if ，暂时不使用

{'uuid': '1a871d01-ed6d-480f-b9e8-c7790320cc92',
 'messages': [{'role': 'system', 'content': ''},
  {'role': 'user',
   'content': "They say you're quick — how fast can you help me find the best Puerto Rican recipes for cooking tonight? I’ve got some fresh ingredients and want to make something flavorful, maybe mofongo or tostones. Got any recommendations?"},
  {'role': 'assistant',
   'content': '¡Ay, bendito! You’re in for a treat — and yes, I’m fast, but flavor can’t be rushed. Luckily, with fresh ingredients and a craving for bold Puerto Rican tastes, we can get you cooking *tonight* with some authentic, crowd-pleasing dishes. Since you mentioned **mofongo** and **tostones**, let’s start there — both are iconic, delicious, and totally doable with fresh plantains (I hope you’ve got green ones for tostones and slightly green-to-yellow for mofongo!).\n\nHere are **two top-tier Puerto Rican recipes** you can make tonight, with room to improvise if you’re feeling creative:\n\n---\n\n###

In [32]:
chat_if_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Instruction-Following-Chat-v1-chat-if.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(chat_if_datas)):
        if elem['thinking'] == True:
            # 判断长度
            cur_length = tokenizer(elem['messages'][1]['content'], return_tensors='pt')['input_ids'].shape[1]
            if cur_length <= 3072:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-if-{}'.format(idx), 'output': elem['messages'][1]['content'].replace('<think>', '').replace('</think>', '').strip()}
            else:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-if-{}'.format(idx), 'output': 'need_235b_distill'}
                errors += 1
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-if-{}'.format(idx), 'output': elem['messages'][1]['content'].strip()}
        chat_if_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
print(errors)

0it [00:00, ?it/s]


KeyError: 'thinking'

In [36]:
struct_output_datas = read_datas(struct_output_path)
print(len(struct_output_datas))

4969


In [37]:
struct_output_datas[0]

{'uuid': 'cd84e0c7-073c-4e6e-b28f-f1ee9c6172f0',
 'messages': [{'role': 'system',
   'content': 'Generate an XML response that strictly conforms to the provided schema, including all required elements, nested structures, data types, and constraints, ensuring accurate representation of attributes, enumerations, and complex types without deviation.'},
  {'role': 'user',
   'content': 'Return an XML response that strictly conforms to the provided SportsCompetitionSummary schema, including all required elements, nested structures, data types, and constraints. Ensure attributes, enumerations, and nested complex types are accurately represented. The response must validate against the schema without deviations.\n\nDocument:\nThe primary event depicted in the image is a professional track and field sprint race, specifically the men\'s 100-meter dash. The scene captures the moment just before the start of the race, with six athletes positioned in numbered lanes, each in a crouched stance behind

In [40]:
struct_output_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Instruction-Following-Chat-v1-struct_output.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(struct_output_datas)):
        if len(elem['messages']) > 3:
            errors += 1
            continue 
            
        # 取出 prompt 和 assistant
        cur_p = ''; cur_a = ''
        for dic_elem in elem['messages']:
            if dic_elem['role'] != 'assistant':
                cur_p += dic_elem['content']
            else:
                cur_a += dic_elem['content']
                break # 只取一轮对话

        # 判断长度
        cur_length = tokenizer(cur_a, return_tensors='pt')['input_ids'].shape[1]
        if cur_length <= 3072:
            cur_ = {'prompt': cur_p,  'source': 'Nemotron-Instruction-Following-Chat-v1-struct_output-{}'.format(idx), 'output': cur_a}
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Instruction-Following-Chat-v1-struct_output-{}'.format(idx), 'output': 'need_235b_distill'}
            errors += 1

        struct_output_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
print(errors)

4969it [00:15, 330.20it/s]

6


In [45]:
# 读取 path 下的所有文件，排除其中 output 是 need_235b_distill 的数据
# 这份数据，答案都是 r1 系列生成的，因此相当于在做数据增强
ori_dir = '/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1'
total_datas = []; errors = 0
for file_name in tqdm(os.listdir(ori_dir)):
    # if file_name == ''
    cur_path = os.path.join(ori_dir, file_name)
    cur_datas = read_datas(cur_path)
    for elem in tqdm(cur_datas):
        if elem['output'] == 'need_235b_distill':
            errors += 1
            continue

        sample = {
            "messages": [
              {
                "role": "user",
                "content": elem['prompt']
              },
              {
                "role": "assistant",
                "content": elem['output']
              }
            ]
        }
    
        total_datas.append(sample)

  0%|          | 0/7 [00:00<?, ?it/s]
0it [00:00, ?it/s]

100%|██████████| 7/7 [25:13<00:00, 216.27s/it]


In [47]:
len(total_datas), errors

(508218, 307728)

In [50]:
math_path = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/math/math_1.jsonl'
general_path = 'code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/general/general_1.jsonl'
swe_t_path = 'code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/swe_testgen.jsonl'

math_datas = read_datas(math_path)

In [51]:
math_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Cascade-SFT-Stage-2-math_1.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(math_datas)):
        if elem['thinking'] == True:
            # 判断长度
            cur_length = tokenizer(elem['messages'][1]['content'], return_tensors='pt')['input_ids'].shape[1]
            if cur_length <= 3072:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-math_1-{}'.format(idx), 'output': elem['messages'][1]['content'].replace('<think>', '').replace('</think>', '').strip()}
            else:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-math_1-{}'.format(idx), 'output': 'need_235b_distill'}
                errors += 1
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-math_1-{}'.format(idx), 'output': elem['messages'][1]['content'].strip()}
        math_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
print(errors, len(math_datas)) # 233046 234641

234641it [5:32:50, 11.75it/s]

233046 234641


In [54]:
general_datas = read_datas(general_path)

In [ ]:
general_datas[0]
print(len(general_datas))
general_datas_deal = []; errors = 0
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/step1/Nemotron-Cascade-SFT-Stage-2-general_1.jsonl', 'w', encoding='utf8') as f_save:
    for idx, elem in tqdm(enumerate(general_datas)):
        if elem['thinking'] == True:
            # 判断长度
            cur_length = tokenizer(elem['messages'][1]['content'], return_tensors='pt')['input_ids'].shape[1]
            if cur_length <= 3072:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-general_1-{}'.format(idx), 'output': elem['messages'][1]['content'].replace('<think>', '').replace('</think>', '').strip()}
            else:
                cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-general_1-{}'.format(idx), 'output': 'need_235b_distill'}
                errors += 1
        else:
            cur_ = {'prompt': elem['messages'][0]['content'], 'source': 'Nemotron-Cascade-SFT-Stage-2-general_1-{}'.format(idx), 'output': elem['messages'][1]['content'].strip()}
        general_datas_deal.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
print(errors, len(general_datas)) # 233046 234641

887614


467217it [41:38, 210.27it/s]

In [49]:
total_datas[0]
save_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/nvidia_stage_2_general_50w.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_datas):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 508218/508218 [07:12<00:00, 1176.02it/s]


In [94]:
# 组合全部 code 数据， code + rw-code + leetcode + rw-leetcode + if 
# 按照 难易程度拼接，中间混合if 最后混合 leetcode 训练


code_path1 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_code_not_over_8_to_10_and_cg_over_8_to_10/Code.jsonl'
code_path2 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10/Code.jsonl'
rw_code_path = '/code/zhaoxudong03/data_pipelines/res_rewrite_training_data_zxd/rw_query_code_265k.jsonl'

leetcode_path = '/code/zhaoxudong03/LeetCodeDataset-train.jsonl'
rw_leetcode_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/leetcode_rw_10_times_gene_res_2507.jsonl'
rw_query_leetcode_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/leetcode_rw_10_times.jsonl'

if_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/IF_188k.jsonl'
 

In [95]:
import os 
import json 
import pandas as pd
from tqdm import tqdm 


In [96]:
# 先训练指令遵循，在训练代码， 
if_datas = []
with open(if_path, 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        if_datas.append({'instruction': cur_['conversations'][0]['content'], 'input': '', 'output': cur_['conversations'][1]['content']})

# 3 epoch 组成一个 epoch， leetcode 题目放在最后，简单题目只在最开始出现（1-3分），

In [4]:
len(if_datas)
# if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110

167338

In [105]:
# 8-10 分难度的没有保存score 字段，随机处理
code_datas = []
with open(code_path2, 'r', encoding='utf8') as f_code1:
    for line in tqdm(f_code1.readlines()):
        cur_ = json.loads(line)
        code_datas.append({'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']})
print(len(code_datas))


100%|██████████| 367104/367104 [00:11<00:00, 32391.16it/s]


367104


In [106]:
code_datas2 = []
with open(code_path1, 'r', encoding='utf8') as f_code:
    for line in f_code.readlines():
        cur_ = json.loads(line)
        code_datas2.append({'instruction': cur_['prompt'], 'input': '', 'output': cur_['output'], 'clarity_score': cur_['score']['Clarity']['Score'], 'cg_score':cur_['score']['Cognitive_Complexity']['Score']})
#         code_datas.append({'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']})

# print(len(code_datas))

In [107]:
len(code_datas)

367104

In [108]:
# 按照认知难度排序，由简单到复杂组装数据
df_code_datas2 = pd.DataFrame(code_datas2)
df_code_datas2 = df_code_datas2.sort_values(by=['cg_score'], ascending=True)

In [109]:
sorted_code_datas2 = []

for idx in range(len(df_code_datas2)):
    sorted_code_datas2.append({'instruction': df_code_datas2.iloc[idx, :]['instruction'], 'input': '', 'output': df_code_datas2.iloc[idx, :]['output']})

In [110]:
code_data_totals = sorted_code_datas2 + code_datas
len(code_data_totals)

798358

In [111]:
code_data_totals_df = pd.DataFrame(code_data_totals)
code_data_totals_df.head()

,instruction,input,output
0,import pytest,,It looks like you've imported the `pytest` fra...
1,"from Crypto.Util.Padding import pad, unpad",,"The line `from Crypto.Util.Padding import pad,..."
2,"给定数组arr，请编写程序计算arr中满足一下条件的i,j的数量",,"请提供具体的条件，以便我能够编写程序计算满足条件的 (i, j) 的数量。\n\n目前您只提..."
3,"from sympy import symbols, I, Matrix, root",,It looks like you've imported several useful c...
4,"from collections import namedtuple, defaultdict",,"The line `from collections import namedtuple, ..."


In [112]:
print(len(code_data_totals_df))
code_data_totals_df = code_data_totals_df.drop_duplicates(subset=['instruction'])
print(len(code_data_totals_df))


798358
798358


In [113]:
# rw-code
rw_code_datas = []
with open(rw_code_path, 'r', encoding='utf8') as f_rw_code_data:
    for line in tqdm(f_rw_code_data.readlines()):
        cur_ = json.loads(line)
        rw_code_datas.append({'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']})


100%|██████████| 265310/265310 [00:08<00:00, 30379.69it/s]


In [114]:
# leetcode
leetcodes = []
dic_= {'Easy': 1, 'Medium': 2, 'Hard':3}
with open(leetcode_path, 'r', encoding='utf8') as f_le:
    for line in tqdm(f_le.readlines()):
        cur_ = json.loads(line)
        leetcodes.append({'instruction': cur_['query'], 'input': '', 'output': cur_['response'], 'score': dic_[cur_['difficulty']]})

100%|██████████| 2641/2641 [00:00<00:00, 6952.71it/s]


In [115]:
# 随机组合数据，训练时开始shuffle
df_leetcodes = pd.DataFrame(leetcodes)
df_leetcodes = df_leetcodes.sort_values(by=['score'])

sorted_leetcodes = []

for idx in range(len(df_leetcodes)):
    sorted_leetcodes.append({'instruction': df_leetcodes.iloc[idx, :]['instruction'], 'input': '', 'output': df_leetcodes.iloc[idx, :]['output']})
# if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110

In [116]:
# rw_leetcode
rw_leetcodes = []
with open(rw_leetcode_path, 'r', encoding='utf8') as f_rw_le:
    for line in f_rw_le.readlines():
        cur_ = json.loads(line)
        rw_leetcodes.append({'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']}) # 'score': df_leetcodes[df_leetcodes['instruction']==cur_['query']]['score']

df_rw_leetcodes = pd.DataFrame(rw_leetcodes)

rw_leetcodes_q = []
with open(rw_query_leetcode_path, 'r', encoding='utf8') as f_q_le:
    for line in f_q_le.readlines():
        cur_ = json.loads(line)
        query = cur_['output_rw']
        if query.startswith('#####改写后的待改写文本开始') and query.endswith('#####改写后的待改写文本结束'):
                rw_q = query.split('#####改写后的待改写文本开始')[1].split('#####改写后的待改写文本结束')[0]
                rw_leetcodes_q.append({"ori_query": cur_['prompt'], "rw_query": rw_q})

df_rw_leetcodes_q = pd.DataFrame(rw_leetcodes_q)

In [117]:
df_rw_leetcodes_merged = pd.merge(df_rw_leetcodes, df_rw_leetcodes_q, left_on=['instruction'], right_on=['rw_query'])
df_rw_leetcodes_merged

,instruction,input,output,ori_query,rw_query
0,\nYou're a skilled Python developer. You'll r...,,```python\nclass Solution:\n def getConcate...,You are an expert Python programmer. You will ...,\nYou're a skilled Python developer. You'll r...
1,\nYou are a skilled Python developer. You'll r...,,```python\nclass Solution:\n def modifyIP(s...,You are an expert Python programmer. You will ...,\nYou are a skilled Python developer. You'll r...
2,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def updateElem...,You are an expert Python programmer. You will ...,\nYou are an expert Python programmer. You wi...
3,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def findDelaye...,You are an expert Python programmer. You will ...,\nYou are an expert Python programmer. You wi...
4,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def prefixCoun...,You are an expert Python programmer. You will ...,\nYou are an expert Python programmer. You wi...
...,...,...,...,...,...
23072,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def sumScores(...,You are an expert Python programmer. You will ...,\nYou are an expert Python programmer. You wi...
23073,\nYou're a skilled Python developer. You'll re...,,```python\nclass Solution:\n def decodeCiph...,You are an expert Python programmer. You will ...,\nYou're a skilled Python developer. You'll re...
23074,\nYou're a skilled Python developer. You'll re...,,```python\nclass Solution:\n def earliestAn...,You are an expert Python programmer. You will ...,\nYou're a skilled Python developer. You'll re...
23075,\nYou are a skilled Python developer. You'll r...,,```python\nclass Solution:\n def minimumCos...,You are an expert Python programmer. You will ...,\nYou are a skilled Python developer. You'll r...


In [118]:
scores = []
for idx in range(len(df_rw_leetcodes_merged)):
    cur_ = df_rw_leetcodes_merged.iloc[idx, :]['ori_query']
    cur_score = df_leetcodes[df_leetcodes['instruction'] == cur_]['score']
    scores.append(cur_score.values.tolist()[0])

In [119]:
df_rw_leetcodes_merged['score'] = scores
df_rw_leetcodes_merged.head()

,instruction,input,output,ori_query,rw_query,score
0,\nYou're a skilled Python developer. You'll r...,,```python\nclass Solution:\n def getConcate...,You are an expert Python programmer. You will ...,\nYou're a skilled Python developer. You'll r...,1
1,\nYou are a skilled Python developer. You'll r...,,```python\nclass Solution:\n def modifyIP(s...,You are an expert Python programmer. You will ...,\nYou are a skilled Python developer. You'll r...,1
2,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def updateElem...,You are an expert Python programmer. You will ...,\nYou are an expert Python programmer. You wi...,2
3,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def findDelaye...,You are an expert Python programmer. You will ...,\nYou are an expert Python programmer. You wi...,1
4,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def prefixCoun...,You are an expert Python programmer. You will ...,\nYou are an expert Python programmer. You wi...,1


In [120]:
df_rw_leetcodes_merged_droped = df_rw_leetcodes_merged[['instruction', 'input', 'output', 'score']]
df_rw_leetcodes_merged_droped

,instruction,input,output,score
0,\nYou're a skilled Python developer. You'll r...,,```python\nclass Solution:\n def getConcate...,1
1,\nYou are a skilled Python developer. You'll r...,,```python\nclass Solution:\n def modifyIP(s...,1
2,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def updateElem...,2
3,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def findDelaye...,1
4,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def prefixCoun...,1
...,...,...,...,...
23072,\nYou are an expert Python programmer. You wi...,,```python\nclass Solution:\n def sumScores(...,3
23073,\nYou're a skilled Python developer. You'll re...,,```python\nclass Solution:\n def decodeCiph...,2
23074,\nYou're a skilled Python developer. You'll re...,,```python\nclass Solution:\n def earliestAn...,3
23075,\nYou are a skilled Python developer. You'll r...,,```python\nclass Solution:\n def minimumCos...,3


In [121]:
df_leetcode_totals = pd.concat([df_rw_leetcodes_merged_droped, df_leetcodes])
df_leetcode_totals = df_leetcode_totals.sort_values(by=['score'])
df_leetcode_totals

,instruction,input,output,score
61,\nYou are a skilled Python developer. You'll r...,,```python\nclass Solution:\n def findColumn...,1
62,\nYou're a skilled Python developer. You'll r...,,```python\nclass Solution:\n def childrenWi...,1
67,\nYou're a skilled Python developer. You'll re...,,```python\nclass Solution:\n def findLonges...,1
68,\nYou're a skilled Python developer. You'll re...,,```python\nclass Solution:\n def orArray(se...,1
69,\nYou are an expert Python programmer. You wil...,,```python\nclass Solution:\n def checkZeroO...,1
...,...,...,...,...
2640,You are an expert Python programmer. You will ...,,"To solve this problem, we need to simulate the...",3
43,You are an expert Python programmer. You will ...,,```python\nclass Solution:\n def isMatch(se...,3
36,You are an expert Python programmer. You will ...,,```python\nclass Solution:\n def solveSudok...,3
50,You are an expert Python programmer. You will ...,,```python\nfrom typing import List\n\nclass So...,3


In [122]:
sorted_leetcodes_totals = []

for idx in range(len(df_leetcode_totals)):
    sorted_leetcodes_totals.append({'instruction': df_leetcode_totals.iloc[idx, :]['instruction'], 'input': '', 'output': df_leetcode_totals.iloc[idx, :]['output']})

In [123]:
sorted_leetcodes_totals[10]
# if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110

{'instruction': "\nYou're a skilled Python developer. You'll receive a task (problem description) and need to write a correct Python program that fulfills the requirements and passes all test cases.\n\n### Problem:\nGiven a positive integer x (in base 10) and a target base b, compute the sum of the digits of x after converting it from base 10 to base b.\nOnce converted, treat each digit as a regular base 10 digit and return their sum in base 10.\n\nExample 1:\n\nInput: x = 34, b = 6  \nOutput: 9  \nExplanation: 34 (base 10) in base 6 is represented as 54. Adding the digits: 5 + 4 = 9.\n\nExample 2:\n\nInput: x = 10, b = 10  \nOutput: 1  \nExplanation: x is already in base 10. The digits are 1 and 0, so 1 + 0 = 1.\n\nConstraints:\n\n1 <= x <= 100  \n2 <= b <= 10\n\n### Format: Use the following template to write your solution and wrap your code within the delimiters.\n```python\nclass Solution:\n    def sumBase(self, x: int, b: int) -> int:\n        \n```\n\n### Response: (use the forma

In [24]:
# 拼装一份数据，不包含rw-code , 只包含 code + if + leetcode + rw-leetcode
concat_code_if_leetcode_totals = if_datas + code_data_totals + sorted_leetcodes_totals 
concat_code_if_leetcode_totals_rw_code = concat_code_if_leetcode_totals + rw_code_datas

In [25]:
len(concat_code_if_leetcode_totals), len(concat_code_if_leetcode_totals_rw_code)

(991414, 1256724)

In [28]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_codes_concat_code_if_leetcode_totals_99w.json', 'w', encoding='utf8') as f_save:
    json.dump(concat_code_if_leetcode_totals, f_save, ensure_ascii=False, indent=4)

In [26]:
len(concat_code_if_leetcode_totals)

991414

In [64]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_codes_all_codes_79w.json', 'w', encoding='utf8') as f_save:
    json.dump(code_data_totals, f_save, ensure_ascii=False, indent=4)

In [30]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_codes_concat_code_if_leetcode_totals_rw_code_125w.json', 'w', encoding='utf8') as f_save:
    json.dump(concat_code_if_leetcode_totals, f_save, ensure_ascii=False, indent=4)

In [39]:
# concat_code_if_leetcode_totals_triple = if_datas*3 + code_data_totals*3 +  
def triple_df_shuffle(df, by_index):
    print(len(df))
    df_triple = pd.concat([df, df, df])
    results = []
    for k, v in df_triple.groupby(by=[by_index]):
        # print(k)
        cur_df = v.sample(frac=1).reset_index(drop=True)
        shuffle_dfs = []
        for idx in tqdm(range(len(cur_df))):
            shuffle_dfs.append({'instruction': cur_df.iloc[idx, :]['instruction'], 'input': '', 'output': cur_df.iloc[idx, :]['output']})

        results.extend(shuffle_dfs)

    print(len(results))
    return results
            

In [40]:
df_code_datas2_triple = triple_df_shuffle(df_code_datas2, 'cg_score')

431254


100%|██████████| 78/78 [00:00<00:00, 12144.77it/s]

1293762


In [41]:
code_datas_triple = code_datas * 3

In [42]:
if_datas_triple = if_datas * 3

In [43]:
df_leetcodes_triple = triple_df_shuffle(df_leetcodes, 'score')

2641


100%|██████████| 1818/1818 [00:00<00:00, 12536.55it/s]

7923


In [44]:
df_rw_leetcodes_triple = triple_df_shuffle(df_rw_leetcodes_merged_droped, 'score')

23077


100%|██████████| 15591/15591 [00:01<00:00, 12503.87it/s]

69231


In [52]:
rw_code_datas_triple = rw_code_datas * 3

In [53]:
print(len(code_datas), len(code_datas_triple))
print(len(if_datas), len(if_datas_triple))
print(len(rw_code_datas), len(rw_code_datas_triple))


367104 1101312
167338 502014
265310 795930


In [56]:
sorted_datas_totals_triple = if_datas_triple + df_code_datas2_triple + code_datas_triple + rw_code_datas_triple + df_rw_leetcodes_triple + df_leetcodes_triple 

In [57]:
len(sorted_datas_totals_triple)

3770172

In [59]:
sorted_datas_totals_triple_NO_IF = df_code_datas2_triple + code_datas_triple + rw_code_datas_triple + df_rw_leetcodes_triple + df_leetcodes_triple 
len(sorted_datas_totals_triple_NO_IF)

3268158

In [58]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_codes_concat_code_if_leetcode_totals_rw_code_sorted_myself_125w_thriple_377w.json', 'w', encoding='utf8') as f_save:
    json.dump(sorted_datas_totals_triple, f_save, ensure_ascii=False, indent=4)

In [60]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_codes_concat_leetcode_totals_rw_code_sorted_myself_thriple_no_IF_326w.json', 'w', encoding='utf8') as f_save:
    json.dump(sorted_datas_totals_triple_NO_IF, f_save, ensure_ascii=False, indent=4)

In [27]:
import json
import os

In [28]:
total_if_datas = []
path_if = '/code/yehangcheng/LLaMA-Factory/data/5-20-188k-if.jsonl'
with open(path_if, 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        sample = {'instruction': cur_['conversations'][0]['content'], 'input': '', 'output': cur_['conversations'][1]['content']}
        total_if_datas.append(sample)

In [29]:
len(total_if_datas)

167338

In [30]:
math_path = '/code/yehangcheng/LLaMA-Factory/data/math-processed-qwen3-235b-ins-res-processed.jsonl'

total_math_datas = []
with open(math_path, 'r', encoding='utf8') as f:
    for line in tqdm(f.readlines()):
        cur_ = json.loads(line)
        sample = {'instruction': cur_['conversations'][0]['content'], 'input': '', 'output': cur_['conversations'][1]['content']}
        total_math_datas.append(sample)

100%|██████████| 591737/591737 [00:12<00:00, 45970.84it/s]


In [36]:
code_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_codes_all_codes_79w.json'
total_code_datas = []
with open(code_path, 'r', encoding='utf8') as f:
    total_code_datas = json.load(f)


In [37]:
len(total_code_datas)

798358

In [100]:
rw_livebench = '/code/zhaoxudong03/Train/LLaMA-Factory/data/zxd/datas/livebench_rewrite_Qwen3_2507/'
rw_livebnech_datas = []
for file_name in os.listdir(rw_livebench):
    if file_name in ['rewrite_datas_2507_gen_ans_data_anaysis.json', 'rewrite_datas_2507_gen_ans_language.json']:
        file_path = os.path.join(rw_livebench, file_name)
        with open(file_path, 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                sample = {'instruction': cur_['instruction'], 'input': '', 'output': cur_['output']}
                rw_livebnech_datas.append(sample)
                

In [101]:
len(rw_livebnech_datas)
# 

32317

In [106]:
import random
rw_livebench = '/code/zhaoxudong03/Train/LLaMA-Factory/data/zxd/datas/livebench_rewrite_Qwen3_2507/'
rw_livebnech_datas_totals = []
for file_name in os.listdir(rw_livebench):
    file_path = os.path.join(rw_livebench, file_name)
    cur_datas = []
    with open(file_path, 'r', encoding='utf8') as f:
        for line in f.readlines():
            cur_ = json.loads(line)
            print(cur_)
            break
    break

{'id': 101, 'instruction': '\n### Instructions: Imagine you\'re a seasoned Python coder.  \nI\'ll hand you a task description plus the opening lines of a Python script that already solves part of it.  \nYour job is to continue from exactly where the snippet stops—just finish writing the rest of the program.  \nDo not include the starting lines again, and do not wrap your extra code in anything other than plain back-ticks.  \nOnly your added portion should appear.\n\n### Task:\nYou’re given a 0-indexed string `text`.  \nIn one move, pick any position `k` and replace `text[k]` with any lowercase English letter.  \nReturn the smallest number of such moves required so that **no two neighboring letters are almost the same**.  \nLetters `a` and `b` are *almost the same* if they are identical or if their ASCII codes differ by exactly 1.\n\nExample 1  \nInput: text = "aaaaa"  \nOutput: 2  \nExplanation: Turn "aaaaa" into "acaca"; the adjacent letters now differ by more than 1. 2 moves are prov

In [34]:
import random
rw_livebench = '/code/zhaoxudong03/Train/LLaMA-Factory/data/zxd/datas/livebench_rewrite_Qwen3_2507/'
rw_livebnech_datas_totals = []
for file_name in os.listdir(rw_livebench):
    file_path = os.path.join(rw_livebench, file_name)
    cur_datas = []
    with open(file_path, 'r', encoding='utf8') as f:
        for line in f.readlines():
            cur_ = json.loads(line)
            sample = {'instruction': cur_['instruction'], 'input': '', 'output': cur_['output']}
            cur_datas.append(sample)
    random.shuffle(cur_datas)
    
    rw_livebnech_datas_totals.extend(cur_datas[:len(cur_datas)//2])
    print(len(rw_livebnech_datas_totals))

5758
13153
27948
36711
54107
62931


In [40]:
total_code_datas[0], len(rw_livebnech_datas_totals)

({'instruction': 'import pytest',
  'input': '',
  'output': "It looks like you've imported the `pytest` framework in Python! `pytest` is a popular testing framework used for writing and running tests in Python. If you'd like help with anything related to `pytest`, such as:\n\n- Writing your first test\n- Using fixtures\n- Running tests\n- Mocking objects\n- Test parametrization\n- Integration with projects\n\nPlease let me know what you're trying to accomplish! For example:\n\n```python\n# Example test using pytest\ndef add(a, b):\n    return a + b\n\ndef test_add():\n    assert add(2, 3) == 5\n    assert add(-1, 1) == 0\n```\n\nYou can run this with:\n```bash\npytest test_file.py\n```\n\nFeel free to ask any specific question!"},
 62931)

In [127]:
zh_110 = []
with open('/code/zhaoxudong03/qwen3_235b_2507_distill_110k.jsonl', 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        # sample = {'instruction': cur_['instruction'], 'input': '', 'output': cur_['output']}
        # sample = {'instruction': cur_['conversations'][0]['content'], 'input': '', 'output': cur_['conversations'][1]['content']}
        sample = {'instruction': cur_['messages'][0]['content'], 'input': '', 'output': cur_['messages'][1]['content']}
        
        # print(sample)
        # print(line)
        zh_110.append(sample)
        # break

In [91]:
total_zhizun_datas = total_if_datas + total_math_datas + total_code_datas + rw_livebnech_datas + zh_110
len(total_zhizun_datas)

total_zhizun_datas_v2 = total_math_datas + total_if_datas + am_thinking_code + zh_110 + rw_livebnech_datas_totals + reason_datas_10w

total_zhizun_datas_v3 = total_math_datas + total_if_datas + code_datas + zh_110 + rw_livebnech_datas_totals + reason_datas_10w


1699750

In [151]:
len(total_math_datas), len(total_if_datas), len(am_thinking_code), len(zh_110), len(rw_livebnech_datas_totals), len(total_code_datas)

(591737, 167338, 292098, 110000, 0, 798358)

In [84]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/totals_zhizun_v1.json', 'w', encoding='utf8') as f_save:
    json.dump(total_zhizun_datas, f_save, ensure_ascii=False, indent=4)

In [92]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/totals_zhizun_v1.jsonl', 'w', encoding='utf8') as f_save:
    for elem in total_zhizun_datas:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [94]:
# 混合 [8,10] 分 code_datas 39w (包括code_datas2 和 2w条数据分析数据), if_datas 18w , leetcode 2w
code_datas_38w_difcult = []
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_code_and_data_analysis.json', 'r', encoding='utf8') as f:
    code_datas_38w_difcult = json.load(f)

In [96]:
concat_code_39w_if_leetcode = if_datas + code_datas_38w_difcult + sorted_leetcodes_totals 

In [99]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_codes_39w_leetcode_totals_IF_59w.json', 'w', encoding='utf8') as f_save:
    json.dump(concat_code_39w_if_leetcode, f_save, ensure_ascii=False, indent=4)

In [126]:
len(concat_code_39w_if_leetcode)

590856

In [49]:
am_thinking_code = []
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd/am-thinking-code-412k.jsonl', 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        if cur_['finish_reason'] == 'stop':
            sample = {'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']}
            am_thinking_code.append(sample)

In [102]:
len(am_thinking_code)
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/am_thinking_code_292k.jsonl', 'w', encoding='utf8') as f_save:
    for elem in am_thinking_code:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [138]:
am_thinking_code_totals = []
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd/am-thinking-code-412k.jsonl', 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        if cur_['finish_reason'] == 'stop':
            sample = {'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']}
            am_thinking_code_totals.append(sample)

In [140]:
len(am_thinking_code_totals)
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/am_thinking_code_389k.jsonl', 'w', encoding='utf8') as f_save:
    for elem in am_thinking_code_totals:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [45]:
reason_datas_10w = []
reason_path = '/code/yehangcheng/Intruct_augment/gen_data/Reasoning/100k.jsonl'
with open(reason_path, 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        sample = {'instruction': cur_['conversations'][0]['content'], 'input': '', 'output': cur_['conversations'][1]['content']}
        reason_datas_10w.append(sample)

In [46]:
len(reason_datas_10w)

100000

In [51]:
# 注意此处 rw_livebench 每个文件随机抽取了一半，累计 6w 条左右
total_zhizun_datas_v2 = total_math_datas + total_if_datas + am_thinking_code + zh_110 + rw_livebnech_datas_totals + reason_datas_10w

In [53]:
len(total_zhizun_datas_v2), len(am_thinking_code)

(1421539, 389533)

In [54]:
len(total_zhizun_datas_v2), len(am_thinking_code)

(1421539, 389533)

In [55]:
len(code_datas)

367104

In [56]:
# v3 将 v2 中的 math 替换成  [8, 10] 分的 39w 条
total_zhizun_datas_v2_1 = total_math_datas + total_if_datas + code_datas + zh_110 + rw_livebnech_datas_totals + reason_datas_10w
len(total_zhizun_datas_v2_1)

1399110

In [113]:
# 126w
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/totals_zhizun_v2.jsonl', 'w', encoding='utf8') as f_save:
    for elem in total_zhizun_datas_v2:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [123]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/totals_zhizun_v2_1_139w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in total_zhizun_datas_v2_1:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [81]:
from tqdm import tqdm 
import json 
import os 

In [130]:
# 汇总所有 [8, 10] 分数据，训练一个基座版本
totals_over_8_to_10 = []
folder_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10'
for file_name in tqdm(os.listdir(folder_path)):
    if file_name.endswith('.jsonl'):
        cur_over_8_to_10 = []
        with open(os.path.join(folder_path, file_name), 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                if cur_['finish_reason'] == 'stop':
                    sample = {'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']}
                    # cur_over_8_to_10.append(sample)
                    totals_over_8_to_10.append(sample)
                    
        # totals_over_8_to_10.append({file_name: cur_over_8_to_10})


100%|██████████| 13/13 [01:26<00:00,  6.64s/it]


In [84]:
len(totals_over_8_to_10)

13

In [93]:
totals_over_8_to_10[0]
for one_cls_ in tqdm(totals_over_8_to_10):
    k = list(one_cls_.keys())[0]
    one_cls = one_cls_[k]
    with open('/code/zhaoxudong03/data_pipelines/training_data/457w/over_8_to_10/{}'.format(k), 'w', encoding='utf8') as f_save:
        for elem in one_cls:
            sample = {
                "messages": [
                  {
                    "role": "user",
                    "content": elem['instruction'] + elem['input']
                  },
                  {
                    "role": "assistant",
                    "content": elem['output']
                  }
                ]
            }
            f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

100%|██████████| 13/13 [05:01<00:00, 23.16s/it]


In [98]:
if_datas[0]

{'instruction': 'Please translate in American English the following Romanian text: “XXX natur"\n',
 'input': '',
 'output': 'XXX natural'}

In [99]:
len(if_datas)
with open('/code/zhaoxudong03/data_pipelines/training_data/457w/IF_datas_167k.jsonl', 'w', encoding='utf8') as f_save:
    for elem in if_datas:
        sample = {
            "messages": [
              {
                "role": "user",
                "content": elem['instruction'] + elem['input']
              },
              {
                "role": "assistant",
                "content": elem['output']
              }
            ]
        }
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

In [104]:
len(rw_livebnech_datas)
with open('/code/zhaoxudong03/data_pipelines/training_data/457w/rw_livebench_two_sub_32k.jsonl', 'w', encoding='utf8') as f_save:
    for elem in rw_livebnech_datas:
        sample = {
            "messages": [
              {
                "role": "user",
                "content": elem['instruction'] + elem['input']
              },
              {
                "role": "assistant",
                "content": elem['output']
              }
            ]
        }
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

In [126]:
len(sorted_leetcodes_totals)
with open('/code/zhaoxudong03/data_pipelines/training_data/457w/sorted_leetcodes_25k.jsonl', 'w', encoding='utf8') as f_save:
    for elem in sorted_leetcodes_totals:
        sample = {
            "messages": [
              {
                "role": "user",
                "content": elem['instruction'] + elem['input']
              },
              {
                "role": "assistant",
                "content": elem['output']
              }
            ]
        }
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

In [129]:
len(zh_110)
with open('/code/zhaoxudong03/data_pipelines/training_data/457w/zh_110k.jsonl', 'w', encoding='utf8') as f_save:
    for elem in zh_110:
        sample = {
            "messages": [
              {
                "role": "user",
                "content": elem['instruction'] + elem['input']
              },
              {
                "role": "assistant",
                "content": elem['output']
              }
            ]
        }
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

In [119]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_all_over_8_to_10_168w.json', 'w', encoding='utf8') as f_save:
    json.dump(totals_over_8_to_10, f_save, ensure_ascii=False, indent=4)

In [ ]:
for k, v in 

In [132]:
len(totals_over_8_to_10), len(if_datas), len(rw_livebnech_datas), len(sorted_leetcodes_totals), len(zh_110)

(1715834, 167338, 32317, 25718, 110000)

In [ ]:

# 整理数据的此处 2020642
filter_all_over_8_to_10_168w_concat_if_leetcode_zh = totals_over_8_to_10 + if_datas  + sorted_leetcodes_totals + zh_110

In [131]:
filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh = totals_over_8_to_10 + if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110
len(filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh)
# 198w 的数据不包含 rw livebench 的 data analysis 和 language 205w应该是多了 over_8_to_10 中数据分析的 3w，之前漏了

2051207

In [ ]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_all_over_8_to_10_168w_concat_if_leetcode_zh_198w.json', 'w', encoding='utf8') as f_save:
    json.dump(filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh, f_save, ensure_ascii=False, indent=4)

In [133]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh_202w.json', 'w', encoding='utf8') as f_save:
    json.dump(filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh, f_save, ensure_ascii=False, indent=4)

In [134]:
len(filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh)

2020642

In [135]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh_202w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [ ]:
total_zhizun_datas = total_if_datas + total_math_datas + total_code_datas + rw_livebnech_datas + zh_110
len(total_zhizun_datas)

total_zhizun_datas_v2 = total_math_datas + total_if_datas + am_thinking_code + zh_110 + rw_livebnech_datas_totals + reason_datas_10w

total_zhizun_datas_v3 = total_math_datas + total_if_datas + code_datas + zh_110 + rw_livebnech_datas_totals + reason_datas_10w

In [144]:
len(total_code_datas), len(rw_livebnech_datas)

(798358, 32317)

In [145]:
len(am_thinking_code), len(rw_livebnech_datas_totals), len(reason_datas_10w)

(292098, 0, 100000)

In [146]:
len(total_zhizun_datas_v2)

1261173

In [155]:
test = total_math_datas + total_if_datas + am_thinking_code + zh_110 + rw_livebnech_datas_totals + reason_datas_10w
len(test)

1324104

In [149]:
len(total_math_datas) + len(total_if_datas) + len(am_thinking_code) + len(zh_110) + len(reason_datas_10w)

1261173

In [150]:
len(rw_livebnech_datas_totals)

0

In [156]:
total_zhizun_datas_v2_gai = total_math_datas + total_if_datas + am_thinking_code + zh_110 + rw_livebnech_datas_totals + reason_datas_10w

total_zhizun_datas_v3_gai = total_math_datas + total_if_datas + code_datas + zh_110 + rw_livebnech_datas_totals + reason_datas_10w

In [157]:
len(total_zhizun_datas_v2_gai), len(total_zhizun_datas_v3_gai)

(1324104, 1399110)

In [158]:
len(total_zhizun_datas_v2), len(total_zhizun_datas_v3)

(1261173, 1336179)

In [161]:
# 126w
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/totals_zhizun_v2_132w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_zhizun_datas_v2_gai):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 1324104/1324104 [15:16<00:00, 1444.88it/s]


In [166]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/totals_zhizun_v3_1_139w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_zhizun_datas_v3_gai):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 1399110/1399110 [20:14<00:00, 1152.00it/s] 


In [163]:

print(len(total_math_datas), len(total_if_datas), len(code_datas), len(zh_110), len(rw_livebnech_datas_totals), len(reason_datas_10w))

591737 167338 367104 110000 62931 100000


In [164]:
len(total_zhizun_datas_v3_gai)

1399110

In [28]:
# 202w + 50w 垂类数据，从sft 开始训练，观察效果
import json 
from tqdm import  tqdm 

def yield_datas(path):
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            yield line

path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh_202w.jsonl'
datas_202w = []
for line in tqdm(yield_datas(path)):
    datas_202w.append(json.loads(line))
    

2020642it [11:56, 2821.60it/s] 


In [29]:
len(datas_202w), datas_202w[0]

(2020642,
 {'instruction': 'Your answer must contain exactly 3 bullet points. Use the markdown bullet points such as: * This is point 1.  * Explain the process of brewing a perfect cup of coffee. * Include the temperature of the water and the recommended brewing time.\n\n\nYou are a malakoff-humanis customer experience expert. Your task is to set the top 30 list of features and best practices that an online customer service must provide to its users for the service to be relevant and efficient.\nOutput should be a table with these informations: Feature or Best Practice name, fulfilled user needs, pros, cons, implementation requirements, estimated time to develop, estimated cost',
  'input': '',
  'output': '* This is point 1.  \n* Explain the process of brewing a perfect cup of coffee.  \n* Include the temperature of the water and the recommended brewing time.'})

In [5]:
domain_150w_path = '/code/jixurui/data/LLM_chuilei/data/意图识别_train_data/train14_domain_150w_base.json'
with open(domain_150w_path, 'r', encoding='utf8') as f_domain_150w:
    domain_150w = json.load(f_domain_150w)

In [7]:
len(domain_150w), domain_150w[0]

general_202w_cat_domain_150w_total_352w = datas_202w + domain_150w
len(general_202w_cat_domain_150w_total_352w)

3519631

In [8]:
from tqdm import tqdm 

with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/general_202w_cat_domain_150w_total_352w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(general_202w_cat_domain_150w_total_352w):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 3519631/3519631 [25:54<00:00, 2264.51it/s] 


In [ ]:
# 处理成 swift 需要的格式
# {"messages": [{"role": "user", "content": "<query1>"}, {"role": "assistant", "content": "<response1>"}]}

2000436 + 20206 

In [2]:
# 随机从 120w 数据中，抽取 50w-70w 业务数据，加入训练
path_domain = '/code/jixurui/data/LLM_chuilei/data/process_data/new_data_sorce_category_label_cursor_postprocess_v3_high_scores_意图_cluster合并_>=0.6/意图识别_train_all/train6_domain_120w.json'
with open(path_domain, 'r', encoding='utf8') as f_domain:
    domain_datas = json.load(f_domain)

In [6]:
len(datas_202w)

NameError: name 'datas_202w' is not defined

In [4]:
import random 
random.shuffle(domain_datas)
domain_50w = domain_datas[:500000]

In [6]:
mix_202w_and_domain_50w = datas_202w + domain_50w
len(mix_202w_and_domain_50w)

2520642

In [7]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/mix_202w_and_domain_50w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in mix_202w_and_domain_50w:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [10]:
len(mix_202w_and_domain_50w)

2520642

In [9]:
tool_datas_path = '/code/yehangcheng/sft_data/Nemotron-Post-Training-Dataset-v1/data/converted_all.jsonl'

for line in yield_datas(tool_datas_path):
    cur_ = json.loads(line)
    print(cur_)
    break

{'messages': [{'role': 'user', 'content': "I'm looking to buy a used truck for my construction work, but I want to make sure it's legitimate. The seller provided the VIN: 1FMXK92W8YPA12345 and said it's registered in Georgia. Can you verify: 1. If the VIN is valid and matches a registered vehicle 2. The registration status in Georgia using the license plate number (if available) 3. Any security interests/liens via PPSR? Let me know if you hit any snags—I’m happy to provide more details if needed."}, {'role': 'assistant', 'content': '<think>\nOkay, let\'s tackle the user\'s request step by step. They want to verify the legitimacy of a used truck they\'re planning to buy. The seller provided the VIN: 1FMXK92W8YPA12345 and mentioned it\'s registered in Georgia. The user has three main questions.\n\nFirst, they want to check if the VIN is valid and matches a registered vehicle. For that, I should use the check_valid_vin function. The VIN is 17 characters, which looks correct, so I\'ll inpu

In [62]:
len(datas_202w)

2020642

In [67]:
## 混合通用 202w 垂类随机抽取 70w 开始训练， 对比单独训练 70w 效果，以及前者在 通用场景的效果变化
domain_120w_path = '/code/jixurui/data/LLM_chuilei/data/process_data/new_data_sorce_category_label_cursor_postprocess_v3_high_scores_意图_cluster合并_>=0.6/意图识别_train_all/train6_domain_120w.json'
domain_over_0428_path = '/workspace/data/data2136/zxd/58_base_data_over_0428_drop_duplicates/Train_Test/train_split.json'


In [22]:
import re
import os
import json
import pandas as pd

def read_datas(path):
    results = []
    try:
        with open(path, 'r', encoding='utf8') as f:
            results = json.load(f)
    except:
        results = []
        with open(path, 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                results.append(cur_)
    return results


In [65]:
domain_120w = read_datas(domain_120w_path)

In [68]:
domain_over_0428 = read_datas(domain_over_0428_path)

In [69]:
len(domain_120w) , len(domain_over_0428)

(1200000, 298989)

In [70]:
deal1_120w = []
for elem in domain_120w:
    deal1_120w.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']})

In [71]:
deal_new = []
for elem in domain_over_0428:
    deal_new.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']})

In [72]:
len(deal1_120w), len(deal_new)

(1200000, 298989)

In [73]:
# 前者抽取 56w 后者 抽取 14w
random.shuffle(deal1_120w)
random.shuffle(deal_new)

domain_before_56w = deal1_120w[:560000]
domain_after_14w = deal_new[:140000]

In [74]:
domain_70w = domain_before_56w + domain_after_14w
general_202w_concat_domain_70w = datas_202w + domain_70w

In [75]:
len(domain_70w), len(general_202w_concat_domain_70w)

(700000, 2720642)

In [77]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/general_202w_concat_domain_70w_total_272w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(general_202w_concat_domain_70w):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 2720642/2720642 [05:15<00:00, 8628.85it/s] 


In [78]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/for_compare_272w_domain_70w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(domain_70w):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 700000/700000 [00:40<00:00, 17130.76it/s]


In [15]:
len(datas_202w)

2020642

In [23]:
import json 
from tqdm import tqdm 

def load_json_or_jsonl(path):
    results = []
    try:
        with open(path, 'r', encoding='utf8') as f:
            results = json.load(f)
    except:
        results = []
        with open(path, 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                results.append(cur_)
    return results

def yield_data(path):
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            yield line

In [2]:
# data=load_json_or_jsonl("/workspace/ye/data/general_202w_concat_domain_70w_total_272w.jsonl")
AceReason = []
for line in tqdm(yield_data('/code/yehangcheng/Intruct_augment/gen_data/AceReason-1.1-SFT/AceReason-1.1-SFT-qwen3-235b-ins.jsonl')):
    cur_ = json.loads(line)
    AceReason.append(cur_)
    
# data1 = load_json_or_jsonl("/code/yehangcheng/Intruct_augment/gen_data/AceReason-1.1-SFT/AceReason-1.1-SFT-qwen3-235b-ins.jsonl")

489970it [28:29, 286.54it/s]


In [3]:
lmsys_1m_clean = []
for line in tqdm(yield_data('/code/yehangcheng/Intruct_augment/gen_data/lmsys_1m_clean/lmsys_1m_clean-qwen3-235b-ins.jsonl')):
    cur_ = json.loads(line)
    lmsys_1m_clean.append(cur_)

273382it [15:19, 297.18it/s]


In [4]:
len(lmsys_1m_clean)

273382

In [5]:
lmsys_1m_clean[0]

{'conversations': [{'role': 'user',
   'content': 'how many floors does the burj al arab have'}],
 'category': 'trivia',
 'outputs': "The Burj Al Arab in Dubai has **28 floors**.  \n\nAlthough it appears taller due to its distinctive sail-shaped structure, the building's interior contains 28 functional floors, including luxury suites, restaurants, and event spaces. The hotel is often described as a 7-star establishment (though this is not an official rating) and stands on its own artificial island, connected to the mainland by a private bridge.",
 'prompt_tokens': 18,
 'completion_tokens': 86}

In [7]:
datas2 = []
for elem in lmsys_1m_clean:
    temp = {'conversations': 
            [
                {'role': 'user', 'content': elem['conversations'][0]['content']},
                {'role': 'assistant', 'content': elem['outputs']}
            ]}
    datas2.append(temp)
    

In [9]:
AceReason[0]

{'conversations': [{'role': 'user',
   'content': '=====Problem Statement=====\nHere is a sample line of code that can be executed in Python:\nprint("Hello, World!")\n\nYou can just as easily store a string as a variable and then print it to stdout:\nmy_string = "Hello, World!"\nprint(my_string)\n\nThe above code will print Hello, World! on your screen. Try it yourself in the editor below!\n\n=====Input Format=====\nYou do not need to read any input in this challenge.\n\n=====Output Format=====\nPrint Hello, World! to stdout.\n\nSolve the problem starting with the provided function header.\n\nFunction header:\n```\nprint("")\n```\n\nPlease place the solution code in the following format:\n```python\n# Your solution code here\n```'}],
 'category': 'code',
 'source': 'apps',
 'outputs': '```python\nprint("Hello, World!")\n```',
 'prompt_tokens': 151,
 'completion_tokens': 11}

In [10]:
datas1 = []
for elem in AceReason:
    temp = {'conversations': 
            [
                {'role': 'user', 'content': elem['conversations'][0]['content']},
                {'role': 'assistant', 'content': elem['outputs']}
            ]}
    datas1.append(temp)

In [12]:
len(datas1), len(datas2)

(489970, 273382)

In [ ]:
{"conversations": [{"role": "user", "content": "Please translate in American English the following Romanian text: “XXX natur\"\n"}, {"role": "assistant", "content": "XXX natural"}]}

In [17]:
datas_202w_cov = []
for elem in datas_202w:
    temp = {'conversations': 
        [
            {'role': 'user', 'content': elem['instruction'] + elem['input']},
            {'role': 'assistant', 'content': elem['output']}
        ]}
    datas_202w_cov.append(temp)

In [18]:
len(datas1), len(datas2), len(datas_202w_cov)

(489970, 273382, 2020642)

In [19]:
general_totals = datas_202w_cov + datas1 + datas2
len(general_totals)

2783994

In [20]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/general_totals_202w_concat_lmsys_278w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in general_totals:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [22]:
len(general_totals)
for elem in general_totals:
    if 'instruction' in elem:
        print(elem)

In [23]:
datas_202w[0]

{'instruction': 'Your answer must contain exactly 3 bullet points. Use the markdown bullet points such as: * This is point 1.  * Explain the process of brewing a perfect cup of coffee. * Include the temperature of the water and the recommended brewing time.\n\n\nYou are a malakoff-humanis customer experience expert. Your task is to set the top 30 list of features and best practices that an online customer service must provide to its users for the service to be relevant and efficient.\nOutput should be a table with these informations: Feature or Best Practice name, fulfilled user needs, pros, cons, implementation requirements, estimated time to develop, estimated cost',
 'input': '',
 'output': '* This is point 1.  \n* Explain the process of brewing a perfect cup of coffee.  \n* Include the temperature of the water and the recommended brewing time.'}

In [27]:
datas2 = []
for elem in lmsys_1m_clean:
    # temp = {'conversations': 
    #         [
    #             {'role': 'user', 'content': elem['conversations'][0]['content']},
    #             {'role': 'assistant', 'content': elem['outputs']}
    #         ]}
    ins_ = elem['conversations'][0]['content']
    inp_ = ''
    out_ = elem['outputs']
    temp = {'instruction': ins_, 'input': '', "output": out_}
    datas2.append(temp)

In [29]:
len(datas2), datas2[0]

(273382,
 {'instruction': 'how many floors does the burj al arab have',
  'input': '',
  'output': "The Burj Al Arab in Dubai has **28 floors**.  \n\nAlthough it appears taller due to its distinctive sail-shaped structure, the building's interior contains 28 functional floors, including luxury suites, restaurants, and event spaces. The hotel is often described as a 7-star establishment (though this is not an official rating) and stands on its own artificial island, connected to the mainland by a private bridge."})

In [30]:
datas1 = []
for elem in AceReason:
    # temp = {'conversations': 
    #         [
    #             {'role': 'user', 'content': elem['conversations'][0]['content']},
    #             {'role': 'assistant', 'content': elem['outputs']}
    #         ]}
    ins_ = elem['conversations'][0]['content']
    inp_ = ''
    out_ = elem['outputs']
    temp = {'instruction': ins_, 'input': '', "output": out_}
    datas1.append(temp)

In [31]:
len(datas1), datas1[0]

(489970,
 {'instruction': '=====Problem Statement=====\nHere is a sample line of code that can be executed in Python:\nprint("Hello, World!")\n\nYou can just as easily store a string as a variable and then print it to stdout:\nmy_string = "Hello, World!"\nprint(my_string)\n\nThe above code will print Hello, World! on your screen. Try it yourself in the editor below!\n\n=====Input Format=====\nYou do not need to read any input in this challenge.\n\n=====Output Format=====\nPrint Hello, World! to stdout.\n\nSolve the problem starting with the provided function header.\n\nFunction header:\n```\nprint("")\n```\n\nPlease place the solution code in the following format:\n```python\n# Your solution code here\n```',
  'input': '',
  'output': '```python\nprint("Hello, World!")\n```'})

In [32]:
general_datas_278w = datas_202w + datas1 + datas2

In [33]:
len(general_datas_278w)

2783994

In [34]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/general_totals_202w_concat_lmsys_278w_one.jsonl', 'w', encoding='utf8') as f_save:
    for elem in general_datas_278w:
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

In [43]:
# 垂类数据中每个文件随机抽取， 构建 垂类任务 baseline 150w 条
path = '/code/jixurui/data/LLM_chuilei/data/process_data/new_data_sorce_category_label_cursor_postprocess_v3_high_scores_意图_cluster合并_>=6/意图识别_train_all/train_all.json'

In [44]:
domain_totals = read_datas(path)

In [101]:
len(domain_totals)

2552292

In [102]:
domain_totals[0]

{'instruction': '角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指导原则：\n---当用户有辱骂情绪，表现为直接骂人说脏话，emotion设置为0，一般包含关键词如：尼玛、垃圾、mlgb、滚、是个屁等，用户仅表达不XX没有骂人的话，不归为该类别；案例：为什么别人虚拟联系我现实诈骗电话->无情感而非辱骂，数次好几个i滚滚滚->辱骂。\n---当用户有急切情绪，表现为嫌弃客服回复慢响应慢，单纯询问时间的不算，emotion设置为1，一般包含关键词如：我很急、着急死了、慢死了、太慢了、效率低等；案例：买会员了什么时候放款->无情感而非急切(用户只是询问某项服务具体时间)，请问排队需要多久时间->急切（用户等待客服服务已经很久）。\n---当用户有失望情绪（不满意、效果差、无体验、感觉麻烦、泄露信息、被骚扰），emotion设置为2，一般包含关键词如：很不满意、太差了、无语、毫无体验、一点用都没有、没效果、不人性化、真麻烦、号码泄露、被骚扰等；案例：VIP没用可以退吗->无情感而非失望（用户仅询问能否退款）。\n---当用户有生气情绪（说骗人、虚假信息、问题未解决、生气），必须表达气愤的情绪，只是反问的话不算，emotion设置为3，一般包含关键词如：卸载、拉黑、瞎搞、骗钱、啥都没解决、坑人、啥都是假的、气死我了、什么玩意、瞎搞、没人管、听不懂人话、太不靠谱了、再也不用了、诈骗等；案例：你不批就拒绝我谢谢 还了你这点钱这辈子不用你们这个平台->生气（很生气说不用平台了）。\n---当用户有投诉情绪，emotion设置为4，一般包含关键词如：我要投诉、12315、315投诉、报警、110等；案例：会员够卖后不成功能退款不->无情感而非投诉（用户仅询问退款问题）。\n---用户无上述情绪指向时，或者用户平静询问问题无激动情绪时，emotion设置为-1；案例：58逾期几天，上催收->无情感（用户只是平静表达情况）。\n限制：\n1、当同时存在多种情绪时，按优先级4、0、3、2、1、-1识别，只输出优先级最高的emotion结果。\n2、不要过度揣测用户意愿，需要根据指导原则给出简短合理解释，放入reason字段。\n3、对异常情绪（emotion为0、1、2、3、4）的判断不能宽泛，需要很严格。\n4、请以json字符

In [47]:
total_domains = []
for elem in domain_totals:
    total_domains.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'], 'file_name':elem['file_name']})

In [50]:
import pandas as pd
df_total_domains = pd.DataFrame(total_domains)

In [51]:
total_domains.head()

,instruction,input,output,file_name
0,角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指...,,"{""emotion"": ""-1"", ""reason"": """"}",311.json
1,角色与目标:\n你是一名金融借款公司客服，现在你需要对用户输入问题进行问答分类。\n指导原则...,,"{""id"": [""6""], ""reason"": ""闲聊""}",311.json
2,角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指...,,"{""emotion"": ""-1"", ""reason"": ""用户平静询问问题，无激动情绪""}",311.json
3,角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指...,,"{""emotion"": ""-1"", ""reason"": ""用户平静询问问题，无激动情绪""}",311.json
4,角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指...,,"{""emotion"": ""-1"", ""reason"": ""用户平静询问问题，无激动情绪""}",265.json


In [52]:
df_total_domain_group = df_total_domains.groupby(['file_name'])

In [103]:
import random 
total_nums = len(total_domains)
total_lst = []
for index, elem in df_total_domain_group:
    # print(index, len(elem))
    cur_nums = len(elem)
    cur_need_nums = int(min(cur_nums, cur_nums / total_nums * 460000))
    cur_list = elem.to_dict('records')
    cur_samples = random.sample(cur_list, cur_need_nums)
    total_lst.extend(cur_samples)
    print(len(cur_samples))
    
    # print(elem)
    # break

7202
2814
8782
36
2095
0
708
24
315
188
236
922
604
1096
314
803
341
289
42
89
127
37
81
86
3797
22
38
102
10
44
57
97
886
15
643
165
314
1425
297
119
111
1371
691
561
15
4718
4715
54
4710
4037
5064
321
4802
3525
3499
10303
3864
150
46
402
1606
594
125
2175
1237
980
1201
2599
82
2573
1291
2683
1248
1197
1038
2606
281
283
665
1314
320
304
5637
17517
5201
590
1304
801
5123
1006
12856
6213
6229
189
12618
23358
39
1187
7758
1077
9990
10297
472
604
1468
5644
303
90
931
891
1049
1015
417
96
14
532
78
1461
1464
10243
9288
28853
20311
869
85
2
832
1901
93
55
4271
105
293
330
7794
3988
2068
5868
2609
54
799
1537
410
23867
18881
32
7
1229
864
876
2886
18340
1032
274
82
7857
5300
186
1885
545
1176
200
85
2512
88
1193
608
123
1588
466
590
587
3759


In [104]:
total_lst[0]

# domain_150w_base = [{'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']} for elem in total_lst]
domain_46w_base = [{'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']} for elem in total_lst]
len(domain_46w_base)

459918

In [63]:
path2 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/IF_188k.jsonl'
path3 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10/Instruction_Following_&_Text_Processing.jsonl'
path4 = '/code/zhaoxudong03/qwen3_235b_2507_distill_110k.jsonl'

gene1_datas = read_datas(path2)
gene2_datas = read_datas(path3)
gene3_datas = read_datas(path4)

In [64]:
len(gene1_datas), len(gene2_datas), len(gene3_datas)

(167338, 21110, 110000)

In [65]:
gene1_datas[0]

{'conversations': [{'role': 'user',
   'content': 'Please translate in American English the following Romanian text: “XXX natur"\n'},
  {'role': 'assistant', 'content': 'XXX natural'}]}

In [66]:
deal1 = []
for elem in gene1_datas:
    deal1.append({'instruction':elem['conversations'][0]['content'], 'input': '', 'output': elem['conversations'][1]['content']})

In [68]:
deal2 = []
for elem in gene2_datas:
    deal2.append({'instruction': elem['prompt'], 'input': '', 'output': elem['output']})

In [81]:
zh_110 = []
for elem in gene3_datas:
    sample = {'instruction': elem['messages'][0]['content'], 'input': '', 'output': elem['messages'][1]['content']}
    zh_110.append(sample)

In [82]:
zh_110[:10]

[{'instruction': '能给我讲一个寓意深刻的故事吗？',
  'input': '',
  'output': '当然可以。下面是一个寓意深刻的小故事，名为《石匠与神像》：\n\n很久以前，在一座深山里，有一位技艺精湛的石匠，名叫阿岩。他一生都在雕刻石头，作品精美绝伦，远近闻名。但他从不雕刻神像，别人问他为什么，他总是沉默不语。\n\n一天，村里要修建一座庙宇，村长请他雕刻一尊守护神像。阿岩犹豫了很久，终于答应了。他选了一块巨大的青石，开始日复一日地凿刻。每一锤、每一凿，他都极为专注，仿佛在与石头对话。\n\n三年过去了，神像终于完成。它面容庄严，目光深邃，仿佛能看透人心。村民们都惊叹不已，纷纷跪拜，称其为“活的神”。\n\n落成典礼那天，村长对阿岩说：“你创造了神，你就是神的化身。”\n\n阿岩却摇了摇头，轻声说：“不，我不是创造者，我只是唤醒了它。”\n\n众人不解。阿岩走到神像前，指着一处细微的裂痕说：“你们看，这块石头原本有一道裂纹，藏在深处。我本想避开它，可无论怎么雕，那裂痕总会出现在神像的眉心。最后我明白——它不是瑕疵，而是它本来的样子。”\n\n他顿了顿，望向群山：“这石头在山中沉睡了千年，经历了风雨雷电，才有了这道伤痕。我所做的，不是塑造完美，而是让伤痕也成为神圣的一部分。”\n\n众人沉默。\n\n阿岩又说：“人总想雕刻出完美的神，可真正的神，不是无瑕的偶像，而是包容伤痕、接纳残缺的存在。我们每个人心中都有一块石头，有裂痕，有沉重，有沉默。但正是这些，让我们真实，让我们值得被唤醒。”\n\n说完，他放下刻刀，走进山林，再也没有回来。\n\n——\n\n**寓意**：  \n这个故事告诉我们，真正的美与力量，不在于完美无缺，而在于接纳自己的伤痕与不完美。每个人都有裂痕，但那不是缺陷，而是经历的印记，是灵魂的深度。我们不必成为别人眼中的“完美神像”，而是要勇敢地“唤醒”真实的自己——带着伤痕，依然庄严站立。'},
 {'instruction': '证明：如果A和B是两个n阶方阵，且AB=BA，那么tr(AB)=tr(BA)。',
  'input': '',
  'output': '我们要证明的是：如果 $ A $ 和 $ B $ 是两个 $ n \\times n $ 的方阵，且 $ AB = BA $，那么  \n$$\n\\te

In [111]:
# domain_base_180w = domain_150w_base + deal1 + deal2 + zh_110
domain_base_76w = domain_46w_base + deal1 + deal2 + zh_110

In [112]:
len(domain_base_76w)

758366

In [94]:
# with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/domain_150w_base_cat_30w_if_total_180w_baseline.json', 'w', encoding='utf8') as f_save:
#     json.dump(domain_base_180w, f_save, ensure_ascii=False, indent=4)

In [114]:
deal_domain_baseline = []
for elem in domain_base_76w:
    if isinstance(elem['output'], str):
        deal_domain_baseline.append(elem)
        

In [115]:
# len(domain_base_180w), len(deal_domain_baseline)
len(domain_base_76w), len(deal_domain_baseline)

(758366, 756943)

In [100]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/domain_150w_base_cat_30w_if_total_180w_baseline.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(deal_domain_baseline):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 1793719/1793719 [01:46<00:00, 16854.58it/s]


In [116]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/domain_46w_base_cat_30w_if_total_76w_baseline.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(deal_domain_baseline):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 756943/756943 [00:50<00:00, 15102.64it/s]


In [7]:

import os 
import json
from tqdm import tqdm 

def read_jsonl(path):
    results = {}
    res = {}; count = 0
    with open(path, 'r', encoding='utf8') as f:
        for line in tqdm(f.readlines()):
            count += 1
            cur_ = json.loads(line)
            # if cur_['finish_reason'] != 'stop':
            #     continue
            res.update({cur_['prompt']: cur_})
        results = res
    return results

In [9]:
# 从 8-10 分的数据中，抽取出 9-10 分的数据，训练观察效果
ori_path_v1 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/clarity_over_9_to_10_and_cg_over_9_to_10'
ori_path_v2 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10'
save_dir = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_9_to_10_and_cg_over_9_to_10'


for file_name in os.listdir(ori_path_v1):
    errors = 0
    try:
        cur_save_path = os.path.join(save_dir, file_name)
        file_path1 = os.path.join(ori_path_v1, file_name)
        file_path2 = os.path.join(ori_path_v2, file_name)
        cur_over_9_to_10 = read_jsonl(file_path1)
        cur_over_8_to_10 = read_jsonl(file_path2)
        with open(cur_save_path, 'w', encoding='utf8') as f_save:
            for k, v in cur_over_9_to_10.items():
                sample = cur_over_8_to_10.get(k, None)
                if not sample:
                    errors += 1
                    continue
                f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')
        print('没有找到的数量为:::', errors)
    except Exception as e:
        print('errors:::::::', e)
        continue

100%|██████████| 43904/43904 [00:00<00:00, 52445.90it/s]


没有找到的数量为::: 0


100%|██████████| 367104/367104 [00:09<00:00, 38264.84it/s]


没有找到的数量为::: 0


100%|██████████| 50676/50676 [00:00<00:00, 54924.94it/s]


...

没有找到的数量为::: 0


100%|██████████| 144048/144048 [00:03<00:00, 39546.02it/s]


没有找到的数量为::: 0


100%|██████████| 1088/1088 [00:00<00:00, 52584.67it/s]

没有找到的数量为::: 0


In [21]:
total_dir = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_9_to_10_and_cg_over_9_to_10'
save_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step3/res_clarity_over_9_to_10_and_cg_over_9_to_10.jsonl'
save_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/res_clarity_over_9_to_10_and_cg_over_9_to_10.jsonl'
res_clarity_over_9_to_10_and_cg_over_9_to_10 = []
nums = 0
# with open(save_path, 'w', encoding='utf8') as f_save:
for file_name in os.listdir(total_dir):
    cur_path = os.path.join(total_dir, file_name)
    with open(cur_path, 'r', encoding='utf8') as f:
        for line in tqdm(f.readlines()):
            sample = json.loads(line)
            cur_ = {'instruction': sample['prompt'], 'input':'', 'output': sample['output']}
            # f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
            res_clarity_over_9_to_10_and_cg_over_9_to_10.append(cur_)
            nums += 1

100%|██████████| 117/117 [00:00<00:00, 23382.74it/s]


In [22]:
nums

863273

In [45]:
len(if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110)
len(rw_livebnech_datas)

32317

In [23]:
# 筛选全部英文数据中 9-10 分 的训练，观察 数据压缩后 效果与 202w 比较，效果是否下降严重
total_res_clarity_over_9_to_10_and_cg_over_9_to_10 = res_clarity_over_9_to_10_and_cg_over_9_to_10 + if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110

In [25]:
len(total_res_clarity_over_9_to_10_and_cg_over_9_to_10)

1198646

In [26]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/total_over_9_to_10_120w.jsonl', 'w', encoding='utf8') as f_save:
    for sample in tqdm(total_res_clarity_over_9_to_10_and_cg_over_9_to_10):
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

100%|██████████| 1198646/1198646 [14:36<00:00, 1367.19it/s]


In [27]:
len(total_res_clarity_over_9_to_10_and_cg_over_9_to_10)

1198646

In [30]:
# datas_202w + 中文 9-10 分 19w， 训练第一版查看效果
total_dir_zh = '/code/zhaoxudong03/data_pipelines/training_data_zxd/wangfei_zh/step2/res_clarity_over_9_to_10_and_cg_over_9_to_10'
res_clarity_over_9_to_10_and_cg_over_9_to_10_zh = []
nums = 0
# with open(save_path, 'w', encoding='utf8') as f_save:
for file_name in os.listdir(total_dir_zh):
    cur_path = os.path.join(total_dir_zh, file_name)
    with open(cur_path, 'r', encoding='utf8') as f:
        for line in tqdm(f.readlines()):
            sample = json.loads(line)
            cur_ = {'instruction': sample['prompt'], 'input':'', 'output': sample['output']}
            # f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
            res_clarity_over_9_to_10_and_cg_over_9_to_10_zh.append(cur_)
            nums += 1
print(len(res_clarity_over_9_to_10_and_cg_over_9_to_10_zh))


100%|██████████| 32/32 [00:00<00:00, 42840.00it/s]

192062


In [32]:
datas_202w_cat_zh_19w_totals_221w = res_clarity_over_9_to_10_and_cg_over_9_to_10_zh + datas_202w

In [33]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/total_202w_cat_zh_19w_total_221w.jsonl', 'w', encoding='utf8') as f_save:
    for sample in tqdm(datas_202w_cat_zh_19w_totals_221w):
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

100%|██████████| 2212704/2212704 [21:15<00:00, 1735.13it/s] 


In [38]:
## 配置一个中文 + 英文 over_9_10 分的观察效果， + zh_100 + if_datas + rw_livebnech_datas + sorted_leetcodes_totals
total_over_9_to_10_139w = res_clarity_over_9_to_10_and_cg_over_9_to_10_zh + total_res_clarity_over_9_to_10_and_cg_over_9_to_10

In [39]:
len(total_over_9_to_10_139w)

1390708

In [40]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/total_over_9_to_10_100w_cat_if_rw_leetcode_total_139w.jsonl', 'w', encoding='utf8') as f_save:
    for sample in tqdm(total_over_9_to_10_139w):
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

100%|██████████| 1390708/1390708 [13:52<00:00, 1670.10it/s]


In [41]:
res_1000 = []
with open('/code/zhaoxudong03/RL/verl_base_zxd/01_llm_releate/data/pretrain.jsonl', 'r', encoding='utf8') as f:
    res_1000 = f.readlines()[:1000]      

In [44]:
with open('/code/zhaoxudong03/RL/verl_base_zxd/01_llm_releate/data/tokenize.jsonl', 'w', encoding='utf8') as f_save:
    for line in res_1000:
        f_save.write(line)

In [1]:
# 中文数据，汇总, 这个over_8_to_10 是排除了全部 over_9_to_10 之后的数据，因此并非真正的 over_8_to_10
path1_zh = '/code/zhaoxudong03/data_pipelines/training_data_zxd/wangfei_zh/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10'
path2_zh = '/code/zhaoxudong03/data_pipelines/training_data_zxd/wangfei_zh/step2/res_clarity_over_9_to_10_and_cg_over_9_to_10'

In [7]:
datas1_zh = []; datas2_zh = []
for file_name in os.listdir(path1_zh):
    try:
        cur_path = os.path.join(path1_zh, file_name)
        cur_datas = read_datas(cur_path)
        # print(cur_datas[0])
        for elem in tqdm(cur_datas):
            datas1_zh.append({'instruction': elem['prompt'], 'input': '', 'output': elem['model_gene']})
        print(f'{file_name} 保存完成')
    except: 
        print(file_name)
        continue

100%|██████████| 68730/68730 [00:00<00:00, 652996.65it/s]


Business_&_Finance.jsonl 保存完成
Code.jsonl


100%|██████████| 95451/95451 [00:00<00:00, 1206125.70it/s]


Creative_Writing.jsonl 保存完成


100%|██████████| 18480/18480 [00:00<00:00, 1145016.37it/s]


...

Others.jsonl 保存完成


100%|██████████| 75905/75905 [00:00<00:00, 1389734.18it/s]


Reasoning_&_Logic.jsonl 保存完成


100%|██████████| 570/570 [00:00<00:00, 1249087.40it/s]

Role_Playing_&_Chat.jsonl 保存完成


In [16]:
print(len(datas1_zh))
# 重新读取 code.jsonl
code_data_zh = []
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd/wangfei_zh/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10/Code.jsonl', 'r', encoding='utf8') as f:
    for line in f:
        elem = json.loads(line)
        code_data_zh.append({'instruction': elem['prompt'], 'input': '', 'output': elem['model_gene']})

1053255


In [19]:
len(code_data_zh)
code_data_zh[0]
datas1_zh.extend(code_data_zh)
len(datas1_zh)

1522950

In [ ]:
for file_name in os.listdir(path2_zh):
    cur_path = os.path.join(path2_zh, file_name)
    cur_datas = read_datas(cur_path)
    # print(cur_datas[0])
    for elem in tqdm(cur_datas):
        datas2_zh.append({'instruction': elem['prompt'], 'input': '', 'output': elem['model_gene']})
    print(f'{file_name} 保存完成')

In [20]:
# 分开保存 
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/total_over_8_to_10_without_over_9_to_10_zh.jsonl', 'w', encoding='utf8') as f_save:
    for sample in tqdm(datas1_zh):
        f_save.write(json.dumps(sample, ensure_ascii=False)+'\n')

100%|██████████| 1522950/1522950 [04:27<00:00, 5697.78it/s] 


In [9]:
import pandas as pd
import random

In [3]:
# 为训练 mini 准备全部高质量训练数据  202w + 70w ic + 70w ner + 30w function call + 30w多轮
# data_domain_ic_150w_path = '/code/jixurui/data/LLM_chuilei/data/意图识别_train_data/train14_domain_150w.json'
data_domain_ic_150w_path = '/code/jixurui/data/LLM_chuilei/data/意图识别_train_data/train28_new_post_progress_domain_150w_add_chatling_merge_ori_235B_response_add_30w_207w.json'


# data_en_general_202w_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/filter_all_over_8_to_10_168w_concat_rw_lb_if_leetcode_zh_202w.jsonl'
ace_reason_path = '/code/zhaoxudong03/zxd_datas//AceReason-1.1-SFT-qwen3-235b-ins.jsonl'  # 50w
# data_domain_ner_150w_path = '/code/jixurui/data/LLM_chuilei/data/槽位提取_train_data/train2_domain_150w.json'
# data_domain_ner_70w_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/domain_ner_70w_from_original_path_drop.jsonl'
data_zh_over_9_to_10_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/round2_zh/res_huizong/datas_zh_over_9_to_10.jsonl'
data_multurn_chat_path_over_8_to_10 = '/code/zhaoxudong03/zxd_datas/multurn/1113_msys_infinity_Magpie_0.8m_349k_prompt_decup_qwen3_235b_2507.jsonl'
data_function_call_path = '/code/zhaoxudong03/zxd_datas/function_call/tool_use_no_think_v2.jsonl'
data_domain_ner_yuhang_path = '/code/zhangyuhang08/model_train/槽位提取_train_data/train2_domain_150w_std_nothink.json'

In [4]:
datas_domain_ic = read_datas(data_domain_ic_150w_path)

In [5]:
datas_domain_ic[0]

{'instruction': '角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指导原则：\n---当用户有辱骂情绪，表现为直接骂人说脏话，emotion设置为0，一般包含关键词如：尼玛、垃圾、mlgb、滚、是个屁等，用户仅表达不XX没有骂人的话，不归为该类别；案例：为什么别人虚拟联系我现实诈骗电话->无情感而非辱骂，数次好几个i滚滚滚->辱骂。\n---当用户有急切情绪，表现为嫌弃客服回复慢响应慢，单纯询问时间的不算，emotion设置为1，一般包含关键词如：我很急、着急死了、慢死了、太慢了、效率低等；案例：买会员了什么时候放款->无情感而非急切(用户只是询问某项服务具体时间)，请问排队需要多久时间->急切（用户等待客服服务已经很久）。\n---当用户有失望情绪（不满意、效果差、无体验、感觉麻烦、泄露信息、被骚扰），emotion设置为2，一般包含关键词如：很不满意、太差了、无语、毫无体验、一点用都没有、没效果、不人性化、真麻烦、号码泄露、被骚扰等；案例：VIP没用可以退吗->无情感而非失望（用户仅询问能否退款）。\n---当用户有生气情绪（说骗人、虚假信息、问题未解决、生气），必须表达气愤的情绪，只是反问的话不算，emotion设置为3，一般包含关键词如：卸载、拉黑、瞎搞、骗钱、啥都没解决、坑人、啥都是假的、气死我了、什么玩意、瞎搞、没人管、听不懂人话、太不靠谱了、再也不用了、诈骗等；案例：你不批就拒绝我谢谢 还了你这点钱这辈子不用你们这个平台->生气（很生气说不用平台了）。\n---当用户有投诉情绪，emotion设置为4，一般包含关键词如：我要投诉、12315、315投诉、报警、110等；案例：会员够卖后不成功能退款不->无情感而非投诉（用户仅询问退款问题）。\n---用户无上述情绪指向时，或者用户平静询问问题无激动情绪时，emotion设置为-1；案例：58逾期几天，上催收->无情感（用户只是平静表达情况）。\n限制：\n1、当同时存在多种情绪时，按优先级4、0、3、2、1、-1识别，只输出优先级最高的emotion结果。\n2、不要过度揣测用户意愿，需要根据指导原则给出简短合理解释，放入reason字段。\n3、对异常情绪（emotion为0、1、2、3、4）的判断不能宽泛，需要很严格。\n4、请以json字符

In [6]:
df_datas_domain_ic = pd.DataFrame(datas_domain_ic)
print('去重前的 shape:::', df_datas_domain_ic.shape)
df_datas_domain_ic_dep = df_datas_domain_ic.drop_duplicates(['instruction'])
print('去重后的 shape:::', df_datas_domain_ic_dep.shape)


去重前的 shape::: (2068357, 3)
去重后的 shape::: (1441779, 3)


In [7]:
# only_one = []
# duplicates = []
# for elem in datas_domain_ic:
#     if elem['instruction'] in only_one:
#         duplicates.append(elem)
#     else:
#         only_one.append(elem['instruction'])

# 新创建一列，== instruction + input 来看去重效果
df_datas_domain_ic['dep'] = df_datas_domain_ic['instruction'] + df_datas_domain_ic['input']
print('去重前的 shape:::', df_datas_domain_ic.shape)
df_datas_domain_ic_dep = df_datas_domain_ic.drop_duplicates(['dep'])
print('去重后的 shape:::', df_datas_domain_ic_dep.shape)

去重前的 shape::: (2068357, 4)
去重后的 shape::: (1958751, 4)


In [9]:
# len(duplicates)
duplicate_mask = df_datas_domain_ic.duplicated(subset=['instruction'], keep=False)
df_duplicates = df_datas_domain_ic[duplicate_mask]

In [10]:
df_datas_domain_ic_dep_lst = df_datas_domain_ic_dep[['instruction', 'input', 'output']].to_dict(orient='records')
random.shuffle(df_datas_domain_ic_dep_lst)
datas_domain_ic_dep_lst_70w = df_datas_domain_ic_dep_lst[:700000]
# df_datas_domain_ic_dep_lst[0]

In [11]:
datas_domain_ic_dep_lst_70w[100]

{'instruction': '',
 'input': '\\n任务描述：\\n你是一个分类器，需要根据用户问题，从提供的候选答案中选择最合适的类别文本，并返回相应结果。你的输出应严格匹配候选答案中的字符串。\\n\\n规则：\\n1.如果没有找到合适的类别文本，返回“拒识”。\\n2.如果问题可以匹配多个类别文本，返回匹配的类别文本，用“||”隔开。\\n3.如果问题唯一匹配一个类别文本，返回该类别文本。\\n\\n注意：\\n1.意图相似度达到80%以上时，可以认为是同一类别文本。\\n2.如果问题涉及辱骂、闲聊、不明确或过于宽泛，返回“拒识”。\\n3.输出一定要和候选答案中的文本完全匹配。\\n4.请不要帮我省略文本，一定要完全输出和候选答案一样的类别文本。\\n\\n输入格式：\\n问题：<用户的问题>\\n候选答案：<类别1>,<类别2>,...,<类别n>\\n\\n输出格式：\\n1.如果匹配多个类别文本，返回“<类别文本1>||<类别文本2>||...||<类别文本n>”。\\n2.如果唯一匹配，返回“<类别文本>”。\\n3.如果拒识，返回“拒识”。\\n\\n现在：\\n用户问题是：为何房东已设置勿扰功能？\\n候选答案是：信息已删除仍会接到电话怎么办｜｜发布房产信息无法输入户型/朝向/楼层等信息怎么办｜｜为什么信息无法修改｜｜为什么被判为房产经纪人｜｜如何防止中介骚扰｜｜怎么取消中介勿扰｜｜面积偏离小区整体范围｜｜为什么拨打房东电话提示中介｜｜如何修改信息\\n输出：\\n',
 'output': '拒识'}

In [12]:
# 读区处理与昂你
datas_domain_ner = read_datas(data_domain_ner_yuhang_path)

In [13]:
print(datas_domain_ner[0])
df_datas_domain_ner = pd.DataFrame(datas_domain_ner)

df_datas_domain_ner['dep'] = df_datas_domain_ner['instruction'] + df_datas_domain_ner['input']
print('去重前的 shape:::', df_datas_domain_ner.shape)
df_datas_domain_ner_dep = df_datas_domain_ner.drop_duplicates(['dep'])
print('去重后的 shape:::', df_datas_domain_ner_dep.shape)

{'instruction': "#角色：你是一个招聘对话分析专家，擅长从对话内容中精准识别招聘者和求职者所具备的各类能力标签，分析时需要严格依据给定的标签定义和匹配规则进行判断，确保识别的准确性和一致性。\n#背景：这是一个人力资源管理系统项目，旨在通过自动化手段对招聘过程中的对话内容进行结构化分析，识别招聘者与求职者各自具备的核心能力标签，从而提升招聘效率和人才匹配精准度。该项目需要从大量文本对话中快速提取符合预设能力维度的信息，为后续的人才筛选和岗位匹配提供数据支持。请根据下面的用户对话提取出招聘者和求职者包含的下列标签中的标签，\n    直接用下面给出的标签名称进行输出，不能输出任何其他标签，标签如下:\n    1. 有招聘经验:如果求职者对话中明确出现了解招聘流程、面试技巧的描述，则认为求职者具备“有招聘经验”标签\n    2. 熟悉办公软件:如果求职者对话中明确出现office、wps、ppt、Excel的描述，则认为求职者具备“熟悉办公软件”标签\n    3. 接待能力:如果求职者对话中明确出现懂接待、待客技巧的描述，则认为求职者具备“接待能力”标签\n    4. 客服能力:如果求职者对话中明确出现理解客户需求、解决客户问题的描述，则认为求职者具备“客服能力”标签\n    5. 沟通协调能力:如果求职者对话中明确出现沟通流畅、协调能力、沟通能力的描述，则认为求职者具备“沟通协调能力”标签\n    6. 团队合作能力:如果求职者对话中明确出现团队中协作、共同实现团队目标的描述，则认为求职者具备“团队合作能力”标签\n    7. 高效执行力:如果求职者对话中明确出现按时完成工作、时间管理的描述，则认为求职者具备“高效执行力”标签\n    8. 资料整理能力:如果求职者对话中明确出现整理资料/文件、管理文件资料的描述，则认为求职者具备“资料整理能力”标签\n    如果是招聘者的对话中出现上述规则中的描述，则同样认为招聘者具备对应标签\n    严格按照示例以json格式输出，多个标签用','分隔开，如果没有标签输出无\n    示例:\n    如果没有任何符合的标签，直接输出:{'招聘者标签':'无','求职者标签':'无'}\n    如果招聘者包含某个标签，求职者不包含任何标签，直接输出:{'招聘者标签':'高效执行力','求职者标

In [19]:
df_datas_domain_ner_dep.head()
df_datas_domain_ner_dep['output'] = df_datas_domain_ner_dep['output'].str.split('</think>').str[-1].str.strip()

/tmp/ipykernel_608661/2288479550.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_datas_domain_ner_dep['output'] = df_datas_domain_ner_dep['output'].str.split('</think>').str[-1].str.strip()


In [22]:
df_datas_domain_ner_dep_lst = df_datas_domain_ner_dep[['instruction', 'input', 'output']].to_dict(orient='records')
random.shuffle(df_datas_domain_ner_dep_lst)
datas_domain_ner_dep_lst_70w = df_datas_domain_ner_dep_lst[:700000]
df_datas_domain_ner_dep_lst[1000]

{'instruction': '#角色：你是一位司机招聘对话信息提取专家，擅长从招聘对话中精准提取应聘者（C端）的关键信息标签，如“司机相关经验”、“驾驶优势”、“驾龄”、“驾照”等，提取时需依据对话上下文判断内容的准确性，并以结构化JSON格式返回结果。\n#背景：该项目是一个针对货运司机岗位的智能招聘系统，旨在通过自动化方式从招聘对话中提取应聘者的相关资质和经验信息，从而提高招聘效率和匹配精准度。系统需要准确识别并结构化处理应聘者的驾驶经验、资质、优势等关键信息，以便招聘方快速评估候选人是否符合岗位要求。从司机招聘对话中提取C端的"司机相关经验"、"驾驶优势"、"驾龄"、"驾照"标签，返回JSON格式:\n\n对话内容为:"""\nB端:你好，我们正在招急聘0元入职帮公司开车，薪资7000-7200，工作地点在渠县-万兴广场-渠江大桥，能求一份您的简历吗\nC端:工作地点在哪里？\nB端:渠县万兴广场渠县人民市场是我们的主要工作地点。\nB端:先为你介绍下哈，工作地点在渠县万兴广场渠县人民市场，月薪范围7000_7200元，包住周末双休包吃年底双薪交通补助加班补助饭补话补房补五险一金。\nB端:您最近是在看工作机会吗？\nB端:您的驾龄是？\nC端:5-10年\nB端:您的驾龄是多久？\nC端:5-10年\nB端:您可以点击下方卡片，我们尽快与您联系进行详细沟通\nC端:我加你微信了\nB端:你可以点击聊天页面顶部投简历或换电话，稍后我会电话联系你。\nC端:简历已投，请多关注，谢谢\nB端:谢谢\nC端:我加你微信了通过一下\n"""\n\nC端标签为:\n',
 'input': '',
 'output': '{\n    "司机相关经验": "",\n    "驾驶优势": "",\n    "驾龄": "5-10年",\n    "驾照": ""\n}'}

In [13]:
datas_202w = read_datas(data_en_general_202w_path)

In [23]:
# datas_domain_ner = read_datas(data_domain_ner_150w_path)
datas_domain_ner = read_datas(data_domain_ner_70w_path)

In [27]:
df_datas_domain_ner = pd.DataFrame(datas_domain_ner)
print('去重前:::', df_datas_domain_ner.shape)
df_datas_domain_ner_dep = df_datas_domain_ner.drop_duplicates(['instruction'])
print('去重后:::', df_datas_domain_ner_dep.shape)
datas_domain_ner_dep_70w = df_datas_domain_ner.to_dict(orient='records')[:700000]
print('长度为：：：', len(datas_domain_ner_dep_70w))

去重前::: (696532, 3)
去重后::: (620358, 3)
长度为：：： 696532


In [23]:
datas_ace_reason = read_datas(ace_reason_path)

In [30]:
datas_domain_ner_70w = datas_domain_ner[:700000]

(700000, 2020642, 700000)

In [24]:
datas_zh_over_9_to_10 = read_datas(data_zh_over_9_to_10_path)

In [25]:
datas_multurn_chat_path_over_8_to_10 = read_datas(data_multurn_chat_path_over_8_to_10)

In [26]:
datas_function_call = read_datas(data_function_call_path)

In [ ]:
datas_domain_ner_dep_lst_70w
datas_domain_ic_dep_lst_70w

In [27]:
# (700000, 2020642, 700000, 489970, 192062, 335170, 310051)
# len(datas_domain_ner_dep_70w), len(datas_202w), len(datas_domain_ic_dep_lst_70w), len(datas_ace_reason), len(datas_zh_over_9_to_10), len(datas_multurn_chat_path_over_8_to_10), len(datas_function_call)
len(datas_domain_ner_dep_lst_70w), len(datas_domain_ic_dep_lst_70w), len(datas_ace_reason), len(datas_zh_over_9_to_10), len(datas_multurn_chat_path_over_8_to_10), len(datas_function_call)

(700000, 700000, 489970, 192062, 335170, 310051)

In [26]:
datas_domain_ner_dep_70w = datas_domain_ner_dep_lst_70w

In [29]:
print(datas_domain_ner_dep_70w[10])
# 抽取 ner 70w 数据组成正常的格式 

datas_domain_ner_dep_70w_format = []
for elem in datas_domain_ner_dep_70w:
    sample = {
        "messages": [
          {
            "role": "user",
            "content": elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    datas_domain_ner_dep_70w_format.append(sample)

{'instruction': '#角色：你是一个地址信息解析专家，擅长从非结构化的地址文本中准确识别并提取出小区名称、楼栋、单元、楼层和房间号等结构化信息，确保提取结果的准确性和一致性。  \n#背景：本项目涉及社区信息的数字化整理工作，旨在对上海市多个小区的房屋地址进行标准化处理，以便后续用于物业管理、数据分析和住户服务等用途。因此需要将自然语言描述的地址信息解析为结构化数据格式。省电信器材场家属院甘肃省兰州市城关区甘南路141～159,27幢,3单元,3层 以community_name、community_address、building_name、units_name、floor_name、house_name为key的json全部输出，其中community_name为小区名称，community_address为小区地址，building_name为楼栋，units_name为单元，floor_name为楼层，house_name为房间，若不包含，则输出“未知”', 'input': '', 'output': '{"community_name":"省电信器材场家属院","community_address":"甘南路141～159","building_name":"27","units_name":"3","floor_name":"3","house_name":"未知"}'}


In [32]:
datas_domain_ner_dep_70w_format[400]

{'messages': [{'role': 'user',
   'content': '#角色：你是一个地址信息解析专家，擅长从非结构化的地址文本中准确识别并提取出小区名称、楼栋、单元、楼层和房间号等结构化信息，确保提取结果的准确性和一致性。  \n#背景：本项目涉及社区信息的数字化整理工作，旨在对上海市多个小区的房屋地址进行标准化处理，以便后续用于物业管理、数据分析和住户服务等用途。因此需要将自然语言描述的地址信息解析为结构化数据格式。江苏省南京市建邺区富春江西街18号融侨中央花园一期6幢1701(普通住宅) 以community_name、community_address、building_name、units_name、floor_name、house_name为key的json全部输出，其中community_name为小区名称，community_address为小区地址，building_name为楼栋，units_name为单元，floor_name为楼层，house_name为房间，若不包含，则输出“未知”'},
  {'role': 'assistant',
   'content': '{"community_name":"融侨中央花园一期","community_address":"江苏省南京市建邺区富春江西街18号","building_name":"6","units_name":"未知","floor_name":"未知","house_name":"1701"}'}]}

In [33]:
# ic 任务数据格式统一
print(datas_domain_ic_dep_lst_70w[0].keys())

datas_domain_ic_dep_lst_70w_format = []
for elem in datas_domain_ic_dep_lst_70w:
    sample = {
        "messages": [
          {
            "role": "user",
            "content": elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    datas_domain_ic_dep_lst_70w_format.append(sample)

dict_keys(['instruction', 'input', 'output'])


In [34]:
datas_domain_ic_dep_lst_70w_format[0]

{'messages': [{'role': 'user',
   'content': "你是一个标记员，下面请根据规则进行标注，判断用户的聊天内容是否违规。\n###澄清定义###\n澄清一:文本只提到加微信/vx/wx/weixin/电话或者手机号码,例如:加13487945687，加个vx，留个微信，微信加一个，我的微233242,加我微q15123063354,加下细聊吧等，输出【通过】。\n澄清二:扣押金/扣费/扣五百/扣掉/抠门/刷扣/扣300/折扣/抠搜等，其中'扣'和'抠'为正常语义，不是代表qq对应的变形字时，需输出【通过】。\n澄清三:如果提及支付宝/抖音，不是添加或者聊天意图，需输出【通过】。\n澄清四:如果提及钉钉子,我不用钉钉,我没有钉钉类似文本，需输出【通过】。\n澄清五:如果提及qq/扣扣,文本内容没有添加或索要qq的意图,例如:为什么加qq/没有qq/不用qq/不用扣扣/谁还用qq/加qq的都是骗子/你qq为什么事危险账号/不加qq/qq农场/QQ大作战等类似文本,需输出【通过】。\n澄清六:如果提及电视/手机投屏，但不涉及微信需用于工作投屏和传输文件，需输出【通过】。\n###违规定义###\n违规一:用户要求添加qq/微信服务号/暴风语/商小信/蔻蔻/微博/钉钉/心桥/wps/土豆/抖音/简讯im/随心聊/闪话/魅信/支付宝/淘宝/skype/海鸥/量子密信/丝瓜/映语通/亚信科技/antmessenger/聚信仁和/飞项/密鸽/随拨等聊天平台，要求下载社交软件，在应用商场下载和搜索软件，或者以工作时间不方便用微信/没有微信/不希望同事看见买房需求/帮领导看房等理由要求添加企鹅/qq/微博/钉钉等聊天软件，都是违规行为，输出【违规】。\n违规二:告诉用户加号为语音通话，要求与对方进行语音沟通，输出【违规】。\n不涉及以上情况的文本输出【通过】。结果只需要输出【通过】或者【违规】即可，以下是需要判断的文本:你好，我个人买房想再详细的了解一下这套房子，您把您的扣 扣或者钉 钉发给我"},
  {'role': 'assistant', 'content': '违规'}]}

In [31]:
# 202w 统一数据格式 
print(datas_202w[0].keys())

datas_202w_format = []
for elem in datas_202w:
    sample = {
        "messages": [
          {
            "role": "user",
            "content": elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    datas_202w_format.append(sample)

dict_keys(['instruction', 'input', 'output'])


In [31]:
# datas_ace_reason 数据格式化 

print(datas_ace_reason[0])
datas_ace_reason_format = []

for elem in datas_ace_reason:
    sample = {
        "messages": [
          {
            "role": "user",
            "content": elem['conversations'][0]['content']
          },
          {
            "role": "assistant",
            "content": elem['outputs']
          }
        ]
    }
    datas_ace_reason_format.append(sample)

{'conversations': [{'role': 'user', 'content': '=====Problem Statement=====\nHere is a sample line of code that can be executed in Python:\nprint("Hello, World!")\n\nYou can just as easily store a string as a variable and then print it to stdout:\nmy_string = "Hello, World!"\nprint(my_string)\n\nThe above code will print Hello, World! on your screen. Try it yourself in the editor below!\n\n=====Input Format=====\nYou do not need to read any input in this challenge.\n\n=====Output Format=====\nPrint Hello, World! to stdout.\n\nSolve the problem starting with the provided function header.\n\nFunction header:\n```\nprint("")\n```\n\nPlease place the solution code in the following format:\n```python\n# Your solution code here\n```'}], 'category': 'code', 'source': 'apps', 'outputs': '```python\nprint("Hello, World!")\n```', 'prompt_tokens': 151, 'completion_tokens': 11}


In [32]:
# len(datas_zh_over_9_to_10), len(datas_multurn_chat_path_over_8_to_10), len(datas_function_call)
# 中文数据格式化
print(datas_zh_over_9_to_10[0])
datas_zh_over_9_to_10_format = []

for elem in datas_zh_over_9_to_10:
    sample = {
        "messages": [
          {
            "role": "user",
            "content":  elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    datas_zh_over_9_to_10_format.append(sample)

{'instruction': '在计算资金成本率时，为什么需要考虑资金的时间价值、投资风险价值和物价变动因素？', 'input': '', 'output': '在计算资金成本率时，需要考虑资金的时间价值、投资风险价值和物价变动因素，主要原因如下：\n\n1. **资金的时间价值（Time Value of Money）**  \n   资金的时间价值是指货币在不同时间点的价值不同，今天的1元钱比未来的1元钱更值钱。这是因为资金可以用于投资并获得收益。在计算资金成本率时，如果不考虑时间价值，就无法准确衡量企业使用资金的真实成本。例如，企业借款后需要在未来偿还本息，这些未来现金流必须通过折现的方式反映其现值，才能与当前获得的资金进行合理比较。因此，使用贴现模型（如资本资产定价模型CAPM或加权平均资本成本WACC）时，时间价值是核心要素。\n\n2. **投资风险价值（Risk Premium）**  \n   不同的资金来源（如股权、债务）和投资项目具有不同的风险水平。投资者要求更高的回报来补偿承担的额外风险。例如，股东承担的风险高于债权人，因此股权资本成本通常高于债务资本成本。在计算资金成本率时，必须根据项目或融资方式的风险程度调整预期回报率。忽略风险因素会导致低估资本成本，进而可能导致企业接受高风险、低回报的项目，损害股东价值。\n\n3. **物价变动因素（通货膨胀影响）**  \n   物价变动，特别是通货膨胀，会降低货币的购买力。如果名义利率未包含对通货膨胀的补偿，实际资金成本会被低估。例如，若贷款利率为6%，而通货膨胀率为3%，则实际利率约为3%。企业在决策时应关注实际资金成本，以确保投资回报能真正覆盖资金使用成本并创造真实收益。因此，在估算资金成本时，需区分名义利率与实际利率，通常通过费雪方程（1 + 名义利率 = (1 + 实际利率) × (1 + 通货膨胀率)）进行调整。\n\n**总结**：  \n综合考虑资金的时间价值、投资风险价值和物价变动因素，有助于企业更准确地衡量资金的真实成本，从而做出科学的投资和融资决策，优化资本结构，提升企业价值。忽略这些因素可能导致资本成本低估，引发过度投资或资源配置不当，增加财务风险。'}


In [33]:
# 多轮对话数据格式化
datas_multurn_chat_path_over_8_to_10_format = []
print(datas_multurn_chat_path_over_8_to_10[0])

for elem in datas_multurn_chat_path_over_8_to_10:
    sample = {
        "messages": elem['conversations']
    }
    datas_multurn_chat_path_over_8_to_10_format.append(sample)

{'conversations': [{'content': '翻译该法律文本，从英语到中文。\nThe parties acknowledge the confidential nature of the information exchanged between them and agree not to disclose it to any third party without the prior written consent of the other party, except in accordance with any legal or regulatory obligation.', 'role': 'user'}, {'content': '双方承认彼此交换信息的保密性质，并同意未经对方事先书面同意，不得向任何第三方披露该信息，但依据任何法律或监管义务披露的除外。', 'role': 'assistant'}, {'content': '现在请你将这段翻译成法语。', 'role': 'user'}, {'content': "Les parties reconnaissent le caractère confidentiel des informations échangées entre elles et s'engagent à ne pas les divulguer à toute tierce partie sans le consentement écrit préalable de l'autre partie, sauf conformément à toute obligation légale ou réglementaire.", 'role': 'assistant'}], 'outputs': '```json\n{\n  "Clarity": {\n    "Reasoning": "指令要求将法律文本从英语翻译为中文，随后再翻译为法语，任务目标明确，上下文完整，未使用模糊词汇或歧义代词。输入文本已提供，翻译格式和语言要求清晰，无缺失关键信息。",\n    "Score": 10\n  },\n  "Cognitive_Complexity": {\n    "Reasoning": "任务要求进行跨语言精确翻译

In [34]:
# datas_function_call 数据格式化
datas_function_call_format = datas_function_call
print(datas_function_call_format[0])

{'messages': [{'role': 'user', 'content': "I'm looking to buy a used truck for my construction work, but I want to make sure it's legitimate. The seller provided the VIN: 1FMXK92W8YPA12345 and said it's registered in Georgia. Can you verify: 1. If the VIN is valid and matches a registered vehicle 2. The registration status in Georgia using the license plate number (if available) 3. Any security interests/liens via PPSR? Let me know if you hit any snags—I’m happy to provide more details if needed."}, {'role': 'tool_call', 'content': '{"name": "check_valid_vin", "arguments": {"vin": "1FMXK92W8YPA12345"}}'}, {'role': 'tool_response', 'content': '{"valid": true, "vin_details": {"make": "Ford", "model": "F-150", "year": 2020, "vehicle_type": "Truck", "registration_status": "Active", "registration_state": "GA", "odometer": 82345, "title_status": "Clear", "lienholder": null, "recall_history": "No active recalls"}, "notes": "VIN is valid and registered in Georgia. PPSR lien check complete - no

In [36]:
# (700000, 2020642, 700000, 489970, 192062, 335170, 310051)
len(datas_domain_ner_dep_70w), len(datas_domain_ic_dep_lst_70w), len(datas_ace_reason), len(datas_zh_over_9_to_10), len(datas_multurn_chat_path_over_8_to_10), len(datas_function_call)

(700000, 700000, 489970, 192062, 335170, 310051)

In [38]:
# (700000, 2020642, 700000, 489970, 192062, 335170, 310051)
len(datas_domain_ner_dep_70w_format), len(datas_domain_ic_dep_lst_70w_format), len(datas_ace_reason_format), len(datas_zh_over_9_to_10_format), len(datas_multurn_chat_path_over_8_to_10_format), len(datas_function_call_format)

(700000, 700000, 489970, 192062, 335170, 310051)

In [39]:
datas_domain_ic_dep_lst_70w_format[0]
datas_domain_ner_dep_70w_format[0]

{'messages': [{'role': 'user',
   'content': '你是一位充满同理心且具备专业判断力的内容审核专家，负责评估文本内容是否违规。\n\n### 审核规则：如果文本内容中包含以下任何违规类型，请输出具体的违规标签并标注出违规片段索引。\n\n#### 违规类型：\n1.**极限词语**：使用极限词，出现"首选"、"唯一"、"独家""最高级""最佳""全国第一"等广告极限词直接形容产品或服务，注意识别变形，如"shou选"、"手选"、"醉好"等。\n2.**时间表示距离**：使用时间描述距离，但并未出现具体距离多少米，如"距离地铁站8分钟"、"小区距离公园仅需10分钟路程"等。\n3.**低首付**：涉及购房低首付的描述，如"0首付"、"0元购"、"0米入住"、"无首付"等描述，首付价格极低以及首付价低于总价的百分之十五，注意识别变形，如"0手扶"、"O艏护"等。\n4.**学区房**：涉及学区房的描述，如"学区房"、"书包房"、"片区房"、"适合家长陪读"，注意单纯描述房源靠近某学校不违规。\n5.**推广联系方式-文本**：询问或者暴露联系方式等，注意识别变形，如"一86联系5三00房东7八96"等。\n6.**税费噱头**：涉及税费的描述，如"无税"、"免税"等。\n7.**未来预期承诺保证**：对未来收益进行承诺或者夸大，如"到手就能盈利"、"回本快"、"5个月回本"等。\n8.**封建迷信**：使用迷信说法进行宣传推销，，如"风水宝地"、"上风上水"、"宝地"等。\n\n### 待审核文本为：业主闲置房产，外地定居，诚心出售此房，房源真实有效，随时配合过户、办手续。\n\n请严格对照上述规则，对文本内容进行细致分析，并输出具体的违规标签和违规片段索引，要求输出为json格式，包含label和position字段，如："{"label":"极限词语","position":"[0,2]"}"。如果文本内容中不包含任何违规类型，请输出"{"label":"无风险违规","position":"[]"}。;'},
  {'role': 'assistant', 'content': '{"label":"无风险违规","position":"[]"}'}]}

In [40]:
total_datas = []
total_datas.extend(datas_domain_ner_dep_70w_format)
# total_datas.extend(datas_202w_format)
total_datas.extend(datas_domain_ic_dep_lst_70w_format)
total_datas.extend(datas_ace_reason_format)
total_datas.extend(datas_zh_over_9_to_10_format)
total_datas.extend(datas_multurn_chat_path_over_8_to_10_format)
total_datas.extend(datas_function_call_format)

In [35]:
save_path = '/code/zhaoxudong03/data_pipelines/training_data/457w/datas_domain_ner_dep_70w_format.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(datas_domain_ner_dep_70w_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 700000/700000 [00:23<00:00, 29334.71it/s]


In [36]:
save_path = '/code/zhaoxudong03/data_pipelines/training_data/457w/datas_domain_ic_dep_lst_70w_format.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(datas_domain_ic_dep_lst_70w_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 700000/700000 [00:43<00:00, 16012.80it/s]


In [44]:
save_path = '/code/zhaoxudong03/data_pipelines/training_data/457w/datas_zh_over_9_to_10_format.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(datas_zh_over_9_to_10_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 192062/192062 [00:30<00:00, 6289.74it/s] 


In [45]:
save_path = '/code/zhaoxudong03/data_pipelines/training_data/457w/datas_multurn_chat_path_over_8_to_10_format.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(datas_multurn_chat_path_over_8_to_10_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 335170/335170 [02:21<00:00, 2362.79it/s]


In [46]:
save_path = '/code/zhaoxudong03/data_pipelines/training_data/457w/datas_function_call_format.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(datas_function_call_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 310051/310051 [01:18<00:00, 3932.64it/s]


In [47]:
save_path = '/code/zhaoxudong03/data_pipelines/training_data/457w/datas_ace_reason_format.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(datas_ace_reason_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 489970/489970 [02:10<00:00, 3763.69it/s]


In [41]:
len(total_datas)

2727253

In [40]:
save_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/total_general_datas_474w_v2.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_datas):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 4744427/4744427 [12:21<00:00, 6402.15it/s] 


In [44]:
random.shuffle(total_datas)

In [45]:
len(total_datas)

4744427

In [82]:
total_datas_no_ace = []
total_datas_no_ace.extend(datas_domain_ner_dep_70w_format)
total_datas_no_ace.extend(datas_202w_format)
total_datas_no_ace.extend(datas_domain_ic_dep_lst_70w_format)
# total_datas.extend(datas_ace_reason_format)
total_datas_no_ace.extend(datas_zh_over_9_to_10_format)
total_datas_no_ace.extend(datas_multurn_chat_path_over_8_to_10_format)
total_datas_no_ace.extend(datas_function_call_format)

In [230]:
len(total_datas_no_ace), len(datas_function_call_format)

(4254457, 310051)

In [85]:
save_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/total_general_datas_425w_del_ace_reason.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_datas_no_ace):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 4254457/4254457 [10:35<00:00, 6698.52it/s] 


In [46]:
save_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/total_general_datas_474w_v2_shuffle.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_datas):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 4744427/4744427 [13:10<00:00, 5999.32it/s]


In [127]:
# 验证 470w 数据中，有多少数据是在 ner 中的
ner_prompt = []
for elem in datas_domain_ner:
    ner_prompt.append(elem['instruction'] + elem['input'])


In [129]:
ner_nums = 0
for elem in total_datas:
    if elem['messages'][0]['content'] in ner_prompt:
        ner_nums += 1

KeyboardInterrupt: 

In [ ]:
ner_nums, total_datas[0]

In [86]:
test_47w = random.sample(total_datas, 470000)

In [87]:
save_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/total_general_datas_474w_random_sample_47w.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(test_47w):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 470000/470000 [04:49<00:00, 1625.94it/s]


In [126]:
len(datas_domain_ner_dep_70w_format), datas_domain_ner_dep_70w_format[100000]

(700000,
 {'messages': [{'role': 'user',
    'content': '金梭路220号9号楼1单元21层94号金梭路220号9号楼1单元21层94号 以community_name、community_address、building_name、units_name、floor_name、house_name为key的json全部输出，其中community_name为小区名称，community_address为小区地址，building_name为楼栋，units_name为单元，floor_name为楼层，house_name为房间，若不包含，则输出“未知”'},
   {'role': 'assistant',
    'content': '{"community_name":"金梭路220号","community_address":"金梭路220号","building_name":"9","units_name":"1","floor_name":"21","house_name":"94"}'}]})

In [93]:
# datas_202w_format 
len(datas_202w_format)

2020642

In [94]:
path_ic_120w = '/code/jixurui/data/LLM_chuilei/data/process_data/new_data_sorce_category_label_cursor_postprocess_v3_high_scores_意图_cluster合并_ge_6/意图识别_train_all/train6_domain_120w.json'
domain_ic_120w = read_datas(path_ic_120w)

In [96]:
len(domain_ic_120w)
domain_ic_120w[0]

{'instruction': '角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指导原则：\n---当用户有辱骂情绪，表现为直接骂人说脏话，emotion设置为0，一般包含关键词如：尼玛、垃圾、mlgb、滚、是个屁等，用户仅表达不XX没有骂人的话，不归为该类别；案例：为什么别人虚拟联系我现实诈骗电话->无情感而非辱骂，数次好几个i滚滚滚->辱骂。\n---当用户有急切情绪，表现为嫌弃客服回复慢响应慢，单纯询问时间的不算，emotion设置为1，一般包含关键词如：我很急、着急死了、慢死了、太慢了、效率低等；案例：买会员了什么时候放款->无情感而非急切(用户只是询问某项服务具体时间)，请问排队需要多久时间->急切（用户等待客服服务已经很久）。\n---当用户有失望情绪（不满意、效果差、无体验、感觉麻烦、泄露信息、被骚扰），emotion设置为2，一般包含关键词如：很不满意、太差了、无语、毫无体验、一点用都没有、没效果、不人性化、真麻烦、号码泄露、被骚扰等；案例：VIP没用可以退吗->无情感而非失望（用户仅询问能否退款）。\n---当用户有生气情绪（说骗人、虚假信息、问题未解决、生气），必须表达气愤的情绪，只是反问的话不算，emotion设置为3，一般包含关键词如：卸载、拉黑、瞎搞、骗钱、啥都没解决、坑人、啥都是假的、气死我了、什么玩意、瞎搞、没人管、听不懂人话、太不靠谱了、再也不用了、诈骗等；案例：你不批就拒绝我谢谢 还了你这点钱这辈子不用你们这个平台->生气（很生气说不用平台了）。\n---当用户有投诉情绪，emotion设置为4，一般包含关键词如：我要投诉、12315、315投诉、报警、110等；案例：会员够卖后不成功能退款不->无情感而非投诉（用户仅询问退款问题）。\n---用户无上述情绪指向时，或者用户平静询问问题无激动情绪时，emotion设置为-1；案例：58逾期几天，上催收->无情感（用户只是平静表达情况）。\n限制：\n1、当同时存在多种情绪时，按优先级4、0、3、2、1、-1识别，只输出优先级最高的emotion结果。\n2、不要过度揣测用户意愿，需要根据指导原则给出简短合理解释，放入reason字段。\n3、对异常情绪（emotion为0、1、2、3、4）的判断不能宽泛，需要很严格。\n4、请以json字符

In [97]:
datas_domain_ic_120w_format = []

for elem in domain_ic_120w:
    sample = {
        "messages": [
          {
            "role": "user",
            "content":  elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    datas_domain_ic_120w_format.append(sample)

In [98]:
concat_202w_domain_ic_120w_total_322w = []
concat_202w_domain_ic_120w_total_322w.extend(datas_202w_format)
concat_202w_domain_ic_120w_total_322w.extend(datas_domain_ic_120w_format)

In [100]:
len(concat_202w_domain_ic_120w_total_322w)
save_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/concat_202w_domain_ic_120w_total_322w.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(concat_202w_domain_ic_120w_total_322w):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 3220642/3220642 [27:24<00:00, 1957.86it/s] 


In [101]:
# 线上 ic 数据去重 
domain_ic_150w_path = '/code/jixurui/data/LLM_chuilei/data/意图识别_train_data/train14_domain_150w.json'
domain_ic_150w = read_datas(domain_ic_150w_path)

In [104]:
domain_ic_150w[0]
domain_ic_150w_com = []
for elem in domain_ic_150w:
    elem['source'] = 'train14_domain_150w'
    domain_ic_150w_com.append(elem)

In [105]:
domain_ic_150w_com[0]

{'instruction': '角色与目标:\n你是一名公司客服，现在你需要对用户输入问题进行情感分析，判断用户情绪。\n指导原则：\n---当用户有辱骂情绪，表现为直接骂人说脏话，emotion设置为0，一般包含关键词如：尼玛、垃圾、mlgb、滚、是个屁等，用户仅表达不XX没有骂人的话，不归为该类别；案例：为什么别人虚拟联系我现实诈骗电话->无情感而非辱骂，数次好几个i滚滚滚->辱骂。\n---当用户有急切情绪，表现为嫌弃客服回复慢响应慢，单纯询问时间的不算，emotion设置为1，一般包含关键词如：我很急、着急死了、慢死了、太慢了、效率低等；案例：买会员了什么时候放款->无情感而非急切(用户只是询问某项服务具体时间)，请问排队需要多久时间->急切（用户等待客服服务已经很久）。\n---当用户有失望情绪（不满意、效果差、无体验、感觉麻烦、泄露信息、被骚扰），emotion设置为2，一般包含关键词如：很不满意、太差了、无语、毫无体验、一点用都没有、没效果、不人性化、真麻烦、号码泄露、被骚扰等；案例：VIP没用可以退吗->无情感而非失望（用户仅询问能否退款）。\n---当用户有生气情绪（说骗人、虚假信息、问题未解决、生气），必须表达气愤的情绪，只是反问的话不算，emotion设置为3，一般包含关键词如：卸载、拉黑、瞎搞、骗钱、啥都没解决、坑人、啥都是假的、气死我了、什么玩意、瞎搞、没人管、听不懂人话、太不靠谱了、再也不用了、诈骗等；案例：你不批就拒绝我谢谢 还了你这点钱这辈子不用你们这个平台->生气（很生气说不用平台了）。\n---当用户有投诉情绪，emotion设置为4，一般包含关键词如：我要投诉、12315、315投诉、报警、110等；案例：会员够卖后不成功能退款不->无情感而非投诉（用户仅询问退款问题）。\n---用户无上述情绪指向时，或者用户平静询问问题无激动情绪时，emotion设置为-1；案例：58逾期几天，上催收->无情感（用户只是平静表达情况）。\n限制：\n1、当同时存在多种情绪时，按优先级4、0、3、2、1、-1识别，只输出优先级最高的emotion结果。\n2、不要过度揣测用户意愿，需要根据指导原则给出简短合理解释，放入reason字段。\n3、对异常情绪（emotion为0、1、2、3、4）的判断不能宽泛，需要很严格。\n4、请以json字符

In [108]:
# 线上 ic 数据读取
ic_chatling_path = '/code/jixurui/data/LLM_chuilei/data/意图分类_chatling_data'
ic_chatling_com = []
for file_name in os.listdir(ic_chatling_path):
    cur_path = os.path.join(ic_chatling_path, file_name)
    cur_datas = read_datas(cur_path)
    cur_datas_com = []
    for elem in cur_datas:
        cur_ = {'instruction': elem['question'], 'input': '', 'output': elem['response'], 'source': file_name.split('.json')[0]}
        cur_datas_com.append(cur_)
    # print(cur_datas_com[0])
    ic_chatling_com.extend(cur_datas_com)
    # break

In [110]:
df_domain_ic_totals = pd.DataFrame(domain_ic_150w_com + ic_chatling_com)

In [113]:
df_domain_ic_totals['dep'] = df_domain_ic_totals['instruction'] + df_domain_ic_totals['input']
df_domain_ic_totals_dep = df_domain_ic_totals.drop_duplicates(subset=['dep'])
df_domain_ic_totals_dep.shape

(1815214, 5)

In [115]:
df_domain_ic_totals.shape

(1832011, 5)

In [120]:
for elem_group in df_domain_ic_totals_dep.groupby('source'):
    print(elem_group[0])
    file_name = elem_group[0]
    cur_dict = elem_group[1].to_dict(orient='records')
    save_dir = '/code/zhaoxudong03/data_pipelines/training_data_zxd/chalting_domain_ic'
    save_path = os.path.join(save_dir, '{}.jsonl'.format(file_name))
    with open(save_path, 'w', encoding='utf8') as f_save:
        for elem in cur_dict:
            f_save.write(json.dumps({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']}, ensure_ascii=False)+'\n')

AI职位帖生成助手
HBG-推荐算法组-顾问服务态度治理
TEG-AI LAB-语音分析-LBG-销售录音意向标签识别和小记
TEG-AI Lab-58主站pc端和wap端智能客服意图分类模型
TEG-AI Lab-HBG-房产技术部-敏感词过滤
TEG-AI Lab-HRG-AI面试间后续问题跳过预测
TEG-AI Lab-HRG-KA新版本大模型对话
TEG-AI Lab-HRG-招聘VIP、招才猫智能客服意图分类
TEG-AI Lab-HRG-招聘极速版智能客服意图分类模型
TEG-AI Lab-智能对话分析-C端透露微信号检测
TEG-AI Lab-智能陪练
TEG-AI Lab-爱房-爱房代理商经纪人通话录音与微聊质检
TEG-AI Lab-语音分析-HRG招聘录音标签识别
TEG-AI Lab-语音分析-通用语音分析场景
TEG-AI Lab-鹊桥-对话四个能力拆解
TEG-AILAB-语音分析-HBG巧房经纪人多跟进记录总结
TEG-AILAB-语音分析-HBG房产销售录音识别和小记
TEG-AILAB-语音分析-HRG实勘职位治理
TEG-AILab-黄页商机行业识别
TEG-信安-L2风险识别（扩召回）-121-三网信审
TEG-信安-免审-121-三网信审
TEG-信安-免审-129-主站二手车
TEG-信安-免审-307-安居客新房运营生产内容
TEG-信安-免审-310-58同镇UGC互动帖审
TEG-搜索推荐部-首页搜索提示-黄页
train14_domain_150w


In [117]:
elem_group

('请选择不正确的陈述。',
        instruction input                                 output  \
 150272  请选择不正确的陈述。        "计算机被用来存储信息和进行计算。不正确的说法是：计算机被用来相互通信。"   
 
                      source         dep  
 150272  train14_domain_150w  请选择不正确的陈述。  )

In [47]:
# DMT 数据组合，采用混合策略两阶段 训练方式 训练
# 以 202w 数据为例子，区分 通用数据  专业数据
# 思路是第一阶段，先训练 专业数据 ['Code.jsonl', 'Data_Analysis.jsonl', 'Math.jsonl', 'Natural_&_Applied_Sciences.jsonl', 'Reasoning_&_Logic.jsonl']
# 第二阶段，从专业数据中抽取 【9-10】 分的混合 通用数据 ['Business_&_Finance.jsonl', 'Other.jsonl', 'Role_Playing_&_Chat.jsonl', 'Creative_Writing.jsonl', 'Daily_Life_&_Health.jsonl', 'General_Knowledge_QA.jsonl', 'Humanities_Arts_&_Social_Sciences.jsonl', 'Instruction_Following_&_Text_Processing.jsonl'] 一起训练

In [48]:
# totals_over_8_to_10 + if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110
# sorted_leetcodes_totals 逻辑稍微复杂，直接在上面完成

In [66]:
# 汇总所有 [8, 10] 分数据，训练一个基座版本
totals_over_8_to_10 = []
profess_over_8_to_10 = []
general_over_8_to_10 = []

profess_type = ['Code.jsonl', 'Data_Analysis.jsonl', 'Math.jsonl', 'Natural_&_Applied_Sciences.jsonl', 'Reasoning_&_Logic.jsonl']
general_type = ['Business_&_Finance.jsonl', 'Other.jsonl', 'Role_Playing_&_Chat.jsonl', 'Creative_Writing.jsonl', 'Daily_Life_&_Health.jsonl', 'General_Knowledge_QA.jsonl', 'Humanities_Arts_&_Social_Sciences.jsonl', 'Instruction_Following_&_Text_Processing.jsonl']
folder_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10'
for file_name in tqdm(os.listdir(folder_path)):
    if file_name.endswith('.jsonl') and file_name in profess_type:
        print(file_name)
        with open(os.path.join(folder_path, file_name), 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                if cur_['finish_reason'] == 'stop':
                    sample = {'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']}
                    profess_over_8_to_10.append(sample)
    else:
        with open(os.path.join(folder_path, file_name), 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                if cur_['finish_reason'] == 'stop':
                    sample = {'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']}
                    general_over_8_to_10.append(sample)

  8%|▊         | 1/13 [00:06<01:13,  6.10s/it]

Code.jsonl


 31%|███       | 4/13 [01:13<02:15, 15.04s/it]

Data_Analysis.jsonl


 62%|██████▏   | 8/13 [01:56<00:50, 10.04s/it]

...

 69%|██████▉   | 9/13 [03:01<01:48, 27.16s/it]

Natural_&_Applied_Sciences.jsonl


 85%|████████▍ | 11/13 [03:59<00:51, 25.97s/it]

Reasoning_&_Logic.jsonl


100%|██████████| 13/13 [04:19<00:00, 19.97s/it]


In [67]:
# if_datas
code_path1 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_code_not_over_8_to_10_and_cg_over_8_to_10/Code.jsonl'
code_path2 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_8_to_10_and_cg_over_8_to_10/Code.jsonl'
rw_code_path = '/code/zhaoxudong03/data_pipelines/res_rewrite_training_data_zxd/rw_query_code_265k.jsonl'

leetcode_path = '/code/zhaoxudong03/LeetCodeDataset-train.jsonl'
rw_leetcode_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/leetcode_rw_10_times_gene_res_2507.jsonl'
rw_query_leetcode_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/leetcode_rw_10_times.jsonl'

if_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/IF_188k.jsonl'

In [68]:
# 指令遵循
if_datas = []
with open(if_path, 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        if_datas.append({'instruction': cur_['conversations'][0]['content'], 'input': '', 'output': cur_['conversations'][1]['content']})

In [69]:
# 改写的livebench 两个子项，数据分析、语言
rw_livebench = '/code/zhaoxudong03/Train/LLaMA-Factory/data/zxd/datas/livebench_rewrite_Qwen3_2507/'
rw_livebnech_datas = []
for file_name in os.listdir(rw_livebench):
    if file_name in ['rewrite_datas_2507_gen_ans_data_anaysis.json', 'rewrite_datas_2507_gen_ans_language.json']:
        file_path = os.path.join(rw_livebench, file_name)
        with open(file_path, 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                sample = {'instruction': cur_['instruction'], 'input': '', 'output': cur_['output']}
                rw_livebnech_datas.append(sample)


In [71]:
# 通用中文数据
zh_110 = []
with open('/code/zhaoxudong03/qwen3_235b_2507_distill_110k.jsonl', 'r', encoding='utf8') as f:
    for line in f.readlines():
        cur_ = json.loads(line)
        # sample = {'instruction': cur_['instruction'], 'input': '', 'output': cur_['output']}
        # sample = {'instruction': cur_['conversations'][0]['content'], 'input': '', 'output': cur_['conversations'][1]['content']}
        sample = {'instruction': cur_['messages'][0]['content'], 'input': '', 'output': cur_['messages'][1]['content']}
        
        # print(sample)
        # print(line)
        zh_110.append(sample)

In [ ]:
# leetcode
leetcodes = []
dic_= {'Easy': 1, 'Medium': 2, 'Hard':3}
with open(leetcode_path, 'r', encoding='utf8') as f_le:
    for line in tqdm(f_le.readlines()):
        cur_ = json.loads(line)
        leetcodes.append({'instruction': cur_['query'], 'input': '', 'output': cur_['response'], 'score': dic_[cur_['difficulty']]})

# 随机组合数据，训练时开始shuffle
df_leetcodes = pd.DataFrame(leetcodes)
df_leetcodes = df_leetcodes.sort_values(by=['score'])

sorted_leetcodes = []

for idx in range(len(df_leetcodes)):
    sorted_leetcodes.append({'instruction': df_leetcodes.iloc[idx, :]['instruction'], 'input': '', 'output': df_leetcodes.iloc[idx, :]['output']})
# if_datas + rw_livebnech_datas + sorted_leetcodes_totals + zh_110

# rw_leetcode
rw_leetcodes = []
with open(rw_leetcode_path, 'r', encoding='utf8') as f_rw_le:
    for line in f_rw_le.readlines():
        cur_ = json.loads(line)
        rw_leetcodes.append({'instruction': cur_['prompt'], 'input': '', 'output': cur_['output']}) # 'score': df_leetcodes[df_leetcodes['instruction']==cur_['query']]['score']

df_rw_leetcodes = pd.DataFrame(rw_leetcodes)

rw_leetcodes_q = []
with open(rw_query_leetcode_path, 'r', encoding='utf8') as f_q_le:
    for line in f_q_le.readlines():
        cur_ = json.loads(line)
        query = cur_['output_rw']
        if query.startswith('#####改写后的待改写文本开始') and query.endswith('#####改写后的待改写文本结束'):
                rw_q = query.split('#####改写后的待改写文本开始')[1].split('#####改写后的待改写文本结束')[0]
                rw_leetcodes_q.append({"ori_query": cur_['prompt'], "rw_query": rw_q})

df_rw_leetcodes_q = pd.DataFrame(rw_leetcodes_q)

df_rw_leetcodes_merged = pd.merge(df_rw_leetcodes, df_rw_leetcodes_q, left_on=['instruction'], right_on=['rw_query'])
df_rw_leetcodes_merged

scores = []
for idx in range(len(df_rw_leetcodes_merged)):
    cur_ = df_rw_leetcodes_merged.iloc[idx, :]['ori_query']
    cur_score = df_leetcodes[df_leetcodes['instruction'] == cur_]['score']
    scores.append(cur_score.values.tolist()[0])

df_rw_leetcodes_merged['score'] = scores
df_rw_leetcodes_merged.head()

df_rw_leetcodes_merged_droped = df_rw_leetcodes_merged[['instruction', 'input', 'output', 'score']]
df_rw_leetcodes_merged_droped

df_leetcode_totals = pd.concat([df_rw_leetcodes_merged_droped, df_leetcodes])
df_leetcode_totals = df_leetcode_totals.sort_values(by=['score'])
df_leetcode_totals

sorted_leetcodes_totals = []

for idx in range(len(df_leetcode_totals)):
    sorted_leetcodes_totals.append({'instruction': df_leetcode_totals.iloc[idx, :]['instruction'], 'input': '', 'output': df_leetcode_totals.iloc[idx, :]['output']})

In [73]:
len(sorted_leetcodes_totals)

25718

In [74]:
# DMT 双阶段混合微调 第一阶段数据组成
# 专业数据 profess_over_8_to_10 + rw_livebnech_datas + sorted_leetcodes_totals
first_round_datas = profess_over_8_to_10 + rw_livebnech_datas + sorted_leetcodes_totals
len(first_round_datas)

1439898

In [77]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/DMT_first_stage_profess_knowledge_144w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(first_round_datas):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 1439898/1439898 [04:02<00:00, 5929.49it/s] 


In [86]:
# first_round swift 格式转换
first_round_datas_format = []

for elem in first_round_datas:
    sample = {
        "messages": [
          {
            "role": "user",
            "content":  elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    first_round_datas_format.append(sample)

In [88]:
len(first_round_datas_format), first_round_datas_format[0]

with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/DMT_first_stage_profess_knowledge_144w_swift.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(first_round_datas_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 1439898/1439898 [04:13<00:00, 5671.30it/s] 


In [89]:
# DMT 双阶段混合微调 第二阶段数据组成
# 专业数据 profess_over_9_to_10 + rw_livebnech_datas + sorted_leetcodes_totals + 通用对话数据 general_over_8_to_10 + if_datas + zh_110 + datas_domain_ner_dep_70w_format + datas_domain_ic_dep_lst_70w_format + datas_zh_over_9_to_10_format + datas_multurn_chat_path_over_8_to_10_format + datas_function_call_format
total_dir = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2/res_clarity_over_9_to_10_and_cg_over_9_to_10'
profess_over_9_to_10 = []
nums = 0
# with open(save_path, 'w', encoding='utf8') as f_save:
for file_name in os.listdir(total_dir):
    if file_name.endswith('.jsonl') and file_name in profess_type:
        print(file_name)
        one_files = []
        cur_path = os.path.join(total_dir, file_name)
        with open(cur_path, 'r', encoding='utf8') as f:
            for line in tqdm(f.readlines()):
                sample = json.loads(line)
                cur_ = {'instruction': sample['prompt'], 'input':'', 'output': sample['output']}
                # f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
                # res_clarity_over_9_to_10_and_cg_over_9_to_10.append(cur_)
                one_files.append(cur_)
        random.shuffle(one_files)
        profess_over_9_to_10.extend(one_files[:len(one_files)//2])

Code.jsonl


100%|██████████| 112177/112177 [00:04<00:00, 22599.59it/s]


Math.jsonl


100%|██████████| 302123/302123 [00:15<00:00, 19721.82it/s]


Natural_&_Applied_Sciences.jsonl


100%|██████████| 284859/284859 [00:09<00:00, 31563.93it/s]


Reasoning_&_Logic.jsonl


100%|██████████| 77999/77999 [00:05<00:00, 14319.94it/s]


In [92]:
# 第二阶段加入的专业知识 为 profess_over_9_to_10 + rw_livebnech_datas + sorted_leetcodes_total 数据量 446612
second_round_1 = profess_over_9_to_10 + rw_livebnech_datas + sorted_leetcodes_totals + general_over_8_to_10 + if_datas + zh_110
len(second_round_1)

# first_round swift 格式转换
second_round_1_format = []

for elem in second_round_1:
    sample = {
        "messages": [
          {
            "role": "user",
            "content":  elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    second_round_1_format.append(sample)

In [94]:
second_round_2_format = datas_domain_ner_dep_70w_format + datas_domain_ic_dep_lst_70w_format + datas_zh_over_9_to_10_format + datas_multurn_chat_path_over_8_to_10_format + datas_function_call_format

In [95]:
len(second_round_1_format), len(second_round_2_format)

(1057921, 2233815)

In [ ]:
# 拼接上online 业务数据 
len(online_datas_457k), online_datas_457k[0]

online_datas_457k_df = pd.DataFrame(online_datas_457k)
online_datas_457k_df['dep'] = online_datas_457k_df['instruction'] + online_datas_457k_df['input']
online_datas_415k_df_dep = online_datas_457k_df.drop_duplicates(['dep'])[['instruction', 'input', 'output']]
print(online_datas_415k_df_dep.shape, len(online_datas_457k))
online_datas_415k_df_dep_lst = online_datas_415k_df_dep.to_dict(orient='records')

online_datas_415k_df_dep_lst_format = []
for elem in online_datas_415k_df_dep_lst:
    sample = {
        "messages": [
          {
            "role": "user",
            "content":  elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    # online_datas_415k_df_dep_lst_format.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
    online_datas_415k_df_dep_lst_format.append(sample)

In [288]:
second_round_datas_format = second_round_1_format + second_round_2_format + online_datas_415k_df_dep_lst_format
len(second_round_datas_format)
random.shuffle(second_round_datas_format)

In [255]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(second_round_datas_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 3707211/3707211 [08:57<00:00, 6890.75it/s]


In [ ]:
len(second_round_datas_format), second_round_datas_format[0]

In [285]:
len(second_round_datas_format)

3707211

In [286]:
# 将function call 数据单独拿出来，防止读取数据出现错误
second_round_2_format_exclude_func_call = datas_domain_ner_dep_70w_format + datas_domain_ic_dep_lst_70w_format + datas_zh_over_9_to_10_format + datas_multurn_chat_path_over_8_to_10_format 

In [289]:
second_round_datas_format_exclude_func_call = second_round_1_format + second_round_2_format_exclude_func_call + online_datas_415k_df_dep_lst_format
len(second_round_datas_format_exclude_func_call)
random.shuffle(second_round_datas_format_exclude_func_call)

In [303]:
DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part1 = []
for elem in second_round_datas_format_exclude_func_call:
    DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part1.append({'messages' : elem['messages']})

In [304]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part1.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part1):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 3397160/3397160 [08:24<00:00, 6730.63it/s]


In [305]:
DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part2 = []
for elem in datas_function_call_format:
    DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part2.append({'messages': elem['messages'], 'tools': elem['tools']})

In [308]:
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part2.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(DMT_second_stage_general_knowledge_cat_online_datas_41w_total_370w_swift_part2):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 310051/310051 [01:32<00:00, 3362.17it/s]


In [ ]:
len(datas_function_call_format)

In [295]:
datas_function_call_format[0].keys()

dict_keys(['messages', 'tools', 'dep'])

In [298]:
second_round_1_format[0].keys()

dict_keys(['messages'])

In [78]:
# 整合全部 ailab 内部的 ic 和 ner 业务数据，加入到 425w 中
# dahao 薪资提取 9k
dahao_xinzi_9k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/training_data_1128.jsonl'
# dahao 二手房 25k
dahao_ershoufang_25k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/training_data_1008.jsonl'
# xudong B端提槽 203k
xudong_b_crf_203k_path = '/code/zhaoxudong03/Train/LLaMA-Factory/data/b_crf_distill_203k.jsonl'
# hangcheng KA 18k
hangcheng_ka_18k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/提问次数_是否回答预测标注_1104_1120_add_special.json'
# xurui ic new 26场景 folder
# xurui_ic_new_26_types_path = '/code/jixurui/data/LLM_chuilei/data/意图分类_chatling_data'
# xurui_ic_new_26_types_path = '/code/jixurui/data/LLM_chuilei/data/process_data/chatling_data_train_test/data_response_before/train_20w_cleaned.jsonl'
xurui_ic_new_26_types_test_path = '/code/jixurui/data/LLM_chuilei/data/process_data/chatling_data_train_test/data_response_before/test_5k_cleaned.jsonl'
xurui_ic_new_26_types_path = '/code/jixurui/data/LLM_chuilei/data/process_data/chatling_data_train_test/data_response_after/train_20w_cleaned_new.jsonl'
xurui_ic_new_26_types_path = '/mnt/code/zhaoxudong03/data_pipelines/training_data/train_20w_cleaned_new.jsonl'

# tianxiang 二分类 modernbert 判断是否支持解答 18k
tianxiang_ic_support_ans_18k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/to_train_01_data.json'
# yuhang 面试间 提槽场景 25k
yuhang_interview_ner_25k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/dataset_2_1120_std.jsonl'
# yuhang agent crm ic 5k
yuhang_agent_crm_ic_5k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/crm_test_data_0626.jsonl'
# feibro C端助手，ic ner 数据
feibro_c_ner_31k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/c_agent_slot_train_data_token_reduction.json'
feibro_c_ic_29k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/c_agent_intention_train_data.json'
# yalong 销售助手槽位提取 4k
yalong_crm_ner_4k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/crm_slot_v5.json'
# yalong ka 陈述句回答的训练数据
yalong_ka_declarative_sentence_qa_6k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/ka_declarative_sentence_qa_train_data_v10.json'
yalong_ka_declarative_sentence_qa_test_path = '/code/zhaoxudong03/data_pipelines/00_yewu/测试数据.xlsx'
# yalong ka bert 模型分类数据，需要自己添加 prompt ，进行训练
yalong_crm_sale_assistant_ic_11k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/crm_train_data_v14.txt'  # 需要自己整理 所有标签，写一个 prompt
yalong_ka_ic_totals_150k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/ka分类模型训练数据.txt'             # 需要自己整理 所有标签，写一个 prompt
# yalong 各个bg 分类数据，bert 类型，需要自己添加 prompt， 进行训练 
yalong_total_bg_ic_781k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/store-only.txt'                    # 需要自己整理 按照第一列 分类后，写一个 prompt , 对每个分类的数据进行抽取组装

# hailong interview 传统模式追问 46k
hailong_interview_zhuiwen_46k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/sft-train1209-1213-1224-0109-0115-0116-0206-0219-0313-0421-0505-0512-0519-0527.json'
# hailong interview 传统模式 提槽 4k
hailong_interview_ner_4k_path = '/code/zhaoxudong03/data_pipelines/00_yewu/ai_interview_dt_v24.json'

In [49]:
dahao_xinzi_9k_ori = read_datas(dahao_xinzi_9k_path)
dahao_xinzi_9k = []
for elem in dahao_xinzi_9k_ori:
    dahao_xinzi_9k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']})
dahao_xinzi_9k[0]

{'instruction': ' 任务是从给定的**职位标题**和**职位内容**中分别提取出**年薪、月薪、日薪、时薪**的信息。\n请根据提供的标题和内容输出如下信息：\n标题年薪，标题月薪，标题日薪，标题时薪，内容年薪，内容月薪，内容日薪，内容时薪。\n\n####\xa0注意事项：\n1.\xa0**标题与内容各自独立提取**，互不依赖。\n2.\xa0**只输出纯数字或“-”**，不添加任何文字描述。\n3.\xa0若数值后带有“万”、“千”等单位，请将其转换为实际金额（如：“15-20万/年”应写为“150000-200000”）。\n4.\xa0若无法识别或没有明确标注薪资，则不输出该项，若标题及内容均无任何薪资描述，则输出{}\n5.\xa0薪资范围使用短横线连接（如“30000-50000”），避免使用中文字符。\n6.\xa0**同单位薪资需综合判断后输出最大范围**。例如：“底薪3000-12000元，表现优秀者可至16000元”\xa0→\xa0输出“3000-16000”。\n7.\xa0输出结果必须严格符合上述格式，不要添加额外说明或解释。注意格式里的双引号，一定要保留对齐原格式，不能做更改\n\n标题：档案管理无需经验待遇优厚6K起\n内容：【岗位职责】<br/>·负责档案录入与整理工作<br/>·协助完成档案资料的归档与管理<br/><br/>【职位要求】<br/>·不限学历，不限经验<br/>·细心、有责任心，能适应档案管理工作<br/><br/>【工作时间】<br/>·具体时间班次面议<br/><br/>【薪资福利】<br/>·月薪5000-8000元<br/>·五险一金|包吃包住|年底双薪|周末双休，有意向请投简历，有意向请投简历，有意向请投简历过来',
 'input': '',
 'output': '{"标题月薪":"6000","内容月薪":"5000-8000"}'}

In [50]:
dahao_ershoufang_25k_ori = read_datas(dahao_ershoufang_25k_path)
random.shuffle(dahao_ershoufang_25k_ori)
dahao_ershoufang_25k_extra_10k = []
for elem in dahao_ershoufang_25k_ori[:10000]:
    dahao_ershoufang_25k_extra_10k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']})

In [51]:
xudong_b_crf_203k_ori = read_datas(xudong_b_crf_203k_path)
random.shuffle(xudong_b_crf_203k_ori)
xudong_b_crf_203k_extra_5w = xudong_b_crf_203k_ori[:50000]

In [52]:
hangcheng_ka_18k_ori = read_datas(hangcheng_ka_18k_path)
hangcheng_ka_18k = []
for elem in hangcheng_ka_18k_ori:
    hangcheng_ka_18k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output']})
hangcheng_ka_18k[0]

{'instruction': '你是一名对话分析助手，任务是根据给定的【历史对话内容】统计【面试官】针对每个面试维度的询问次数。\n\n    请严格按照以下规则与步骤执行：\n\n    一、计数对象与基本原则\n\n    1. 只统计“面试官”在对话中**直接向求职者提出的询问或请求确认**。只有面试官发出的询问才计数，求职者的提问或回复不计入面试官的询问次数。\n    2. **岗位描述 / 帖子信息 / 面试官的宣告性要求（例如“本职位要求54岁以下”或“待遇7000-10000”）不会自动计为询问**，除非该句以问题形式或明确向求职者发出确认（例如“您符合54岁以下的要求吗？”），才视为一次询问。\n    3. 计数范围严格为 0～3 次：同一维度被询问 0 次记 0，询问 1 次记 1，……询问 ≥3 次一律记 3（即上限为 3）。\n    4. 第一段面试官的开场白介绍岗位相关信息要求投递简历、交换微信、交换手机号，不能放入维度数量的统计中：\n    例1:\n        询问维度：\n            交换微信\n        历史对话：\n            【面试官】你好，我们还在招聘的，工作内容都比较简单，主要是分类整理这些，保底6000以上，工作简单，易上手，都是手上活哦亲❤????【换行】【换行】发面试邀请投递换微信。\n            【求职者】xxx。\n        以上的情况交换微信维度询问次数为0，因为这是一个开场白，不要放入在维度统计中。\n    例2:\n        询问维度：\n            1.年龄 2.性别 3.投递简历\n        历史对话：\n            【求职者】：enter_im\n            【面试官】：我们这边正在招55岁以下的阿姨来接送小孩，工作时间是早9点到晚5点，薪资范围在8000到12000之间，包吃包住，还有五险一金和周末双休呢，有兴趣的话可以来聊聊吗？\n            【面试官】：我们这边正在招55岁以下的阿姨来接送小孩，工作时间早9点到晚5点，薪资8000到11000，包吃包住，还有五险一金，挺不错的，有兴趣的话可以聊聊吗？\n            【求职者】：是哪里上班呢\n            

In [63]:
# 旭瑞 26 个新场景数据抽取，20w 条
xurui_ic_new_26_types_20w = read_datas(xurui_ic_new_26_types_path)

In [68]:
tianxiang_ic_support_ans_18k_ori = read_datas(tianxiang_ic_support_ans_18k_path)
tianxiang_ic_support_ans_18k = []
for elem in tianxiang_ic_support_ans_18k_ori:
    tianxiang_ic_support_ans_18k.append({'instruction': elem['sentence'], 'input': '', 'output': elem['label']})
tianxiang_ic_support_ans_18k[0]

{'instruction': '玲珑骰子安红豆的寓意[SEP]{"内容_1": "在空白页的最后一个位置点鼠标, 然后按退格键.直到到达你文章的最后页.\\n空白页删除不了，可能是你在空白页里有东西，比如说  回车、 空格。\\n把视图切换到普通视图，看看最后一页是不是有分节符之类的东西，影响到删除不了空白纸。有的话直接删除就可以了。\\n把光标停留在空白页，按住Backspce，删掉空格就好。\\n<p><img src=\\"37250398608\\" /></p>\\nWord怎么删除空白页方法汇总1.将鼠标放在前一页的最后,用DEL健删除。如果空白面是最后一页,且鼠标在第一行，可选“格式”-“段落”，将这一行的行距设为固定值1磅，该空白页将自动消失。2.先显示分页符，即在Word的左下角调整到“普通视图”状态，这时分页符就出现了，直接删除即可。 3 选择“替换”点“高级”，在里面选择“使用通配符”以后下面有一个“特殊字符”字的开头，按住shift的时候再点下鼠标，选择空白页,再删除（解决了我的问题）4.如果是插入分页符造成的空白页，少的话，删除分页符就行，就是到空白页顶部按退格键。（普通视图下或打开编辑标记会显示分页符） 5.如果分页符很多，可以编辑/替换/高级/特殊字符/人工分页符/全部替换就可以了。6.如果是你画了一个表格，占了一整页，造成最后一个回车在第二页删不了，可以将表格缩小一点或者将上面或者下面页边距设小一点，在文件/页面设置中，上下的数字改小一点。7.将鼠标放在前一页的最后,用DEL健删除。如果空白面是最后一页,且鼠标在第一行，可选“格式”-“段落”，将这一行的行距设为固定值1磅，该空白页将自动消失。8、后面有空白是上一页内容过多导致的，一般可以把鼠标点到空白面上，然后按回退键，退有内容的那一面，空白的就没有了，如果还存在，可以稍调整一下上一页内容，少一行就可以了 。9、word 预览有空白页 页面视图时没有。空白页有页码，造成我打印的文档页码不连续。怎样删除：可能是你的文档中有过宽，过长的对象（如表格，图片，公式），导致与打印纸张的规格不一至，调整附近的对象（如表格，图片，公式）大小看看。也可能与分栏和一些可个和回车符号有关。10、ctrl+enter即可去除空白页11、插入表格后的Word删除空白页     在Word2003中插入一张

In [67]:
yuhang_interview_ner_25k_ori = read_datas(yuhang_interview_ner_25k_path)
yuhang_interview_ner_25k = []
for elem in yuhang_interview_ner_25k_ori:
    yuhang_interview_ner_25k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
yuhang_interview_ner_25k[0]

{'instruction': '\n角色：你是一个具有世界知识和日常工作生活常识的信息提取专家，你的主要任务是根据要求从求职者与AI助手之间的历史对话中提取关键信息\n你需要根据对话来提取面试者的关键信息，然后以json的格式进行返回\n我将提供给你面试者和AI面试官之间直到当前轮的对话，之后你需要识别出其中所有的关键信息\n今天的具体日期是:2025/9/21周日，后面你可能需要据此来判断求职者的年龄、最快入职时间等信息。\n已知当前面试的岗位是：保洁,\n【关键信息】如下，包括：\n##年龄##\n###回答格式说明###\n{X}岁\n###回答呈现格式###\nint值，阿拉伯数字格式\n注意不要遗漏单位：岁\n模糊描述处理：\na. 类似\'刚成年\'\'刚大学毕业\'等有普遍年龄区间描述的，映射成具体值如\'16岁\'\'23岁\'...\nb. 面试者描述出生年月的如\'82年的\'，需计算当前年-出生年份的具体值，即当前年份2025-1982，提取43岁\n面试者描述出生年月的如\'10年的\'，需计算当前年-出生年份的具体值，即当前年份2025-2010，提取15岁\nc. 类似\'30多岁\'\'20多一点\'等，映射成\'31\'\'21\'...\nd. 描述多个年龄如 \'17岁。57岁\'，取最后一个年龄，即 57岁\ne. 对于 \'不满二十\'\'不到二十\' 等词，年龄向下取一岁，例如 \'19岁\'\n##身高##\n###回答格式说明###\n{X}cm\n###回答呈现格式###\nint值，阿拉伯数字格式\n注意不要遗漏单位：cm\n模糊描述处理：\na. 其他单位如米、公分等，需自动进行映射\nb. 类似\'一米七多\'\'不到一米六\'等，向下取成\'170cm\'\'159cm\'\nc. 如果面试者说的身高值模糊不清，例如：172 3cm。。。需要输出172cm\nd. 如果面试者说了多个值：“差不多182、183cm这样”，取最高值输出：183cm\n##体重##\n###回答格式说明###\n{X}斤\n###回答呈现格式###\nint值，阿拉伯数字格式\n注意不要遗漏单位：斤\n模糊描述处理：\na. 其他单位如公斤、千克等，需自动进行映射\nb. 类似\'一百多一点\'\'不到一百六\'等，向下取成\'1

In [66]:
yuhang_agent_crm_ic_5k = read_datas(yuhang_agent_crm_ic_5k_path)

In [65]:
feibro_c_ner_31k_ori = read_datas(feibro_c_ner_31k_path)
feibro_c_ner_31k = []
for elem in feibro_c_ner_31k_ori:
    feibro_c_ner_31k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
feibro_c_ner_31k[0]

{'instruction': '根据对话内容，填充所需要的字段。需要填充的字段如下：\n###\n{\n    "姓名": "",\n    "性别": "",\n    "出生年份:" "",\n    "年龄": "",\n    "学历": "",\n    "期望城市": "",\n    "非期望城市": "",\n    "期望职位": "",\n    "非期望职位": "",\n    "期望薪资": "",\n    "工作场所": "",\n    "企业或品牌名称": "",\n    "工作性质": "",\n    "工资结算方式": ""\n}\n###\n\n填充时遵循如下规则：\n1、如果对话中未明确提到信息，则认为提取不到（未知），结果中不需要出现对应的键值对。\n2、年龄应该输出一个阿拉伯数字，如果求职者说了出生年份则应该通过出生年份转换为年龄，今年是2024年。\n3、期望薪资也应尽可能转换为整数数字或者数字区间（例如1000-2000），如果求职者期望面议则填充"面议"。\n4、学历应当归于下列选项之一：["高中以下","高中","中专/技校","大专","本科","硕士","博士","MBA/EMBA"]。\n5、工作年限应当转换为整数（向下取整），不要根据对话中的年份信息进行推断。如果明确提到无经验（没做过），则应当填充"无经验"；如果是应届生（刚毕业、毕业不久等），则应当填充"应届生"；如果是还没毕业（还是学生、还没毕业等），则应当填充"在校生"。\n6、期望职位仅填写职位名，不要填写额外的内容，如果提到多个，则全部输出，用“/”进行分割。不要把求职者的过往工作当成期望职位。\n7、非期望职位是用户明确说不想从事的工作，如果用户说不想做某工作则填写，如果非期望职位有多个，则全部输出，用“/”分割。\n8、如果期望城市有多个，则全部输出，用“/”进行分割。\n9、非期望城市是用户明确说不想工作的城市，如果用户说不想在某个地方找工作则填写，如果非期望城市有多个，全部输出，用“/”分割。\n10、工作场所通常用来指代某些类型的工作，例如："电子厂"、"工地"、"超市"等词语，不是城市、区县这样的地点概念，也不是直接表达的职位名称。如果提到多个则用“/”进行分割，如果未提到则填为null。\n11、企业或品牌名称是指具体的企业

In [64]:
feibro_c_ic_29k_ori = read_datas(feibro_c_ic_29k_path)
feibro_c_ic_29k = []
for elem in feibro_c_ic_29k_ori:
    feibro_c_ic_29k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
feibro_c_ic_29k[0]

{'instruction': '你现在扮演一个求职助手，帮助用户写简历和找工作。通过对话内容判断用户的意图是什么。意图如下，每行是一个意图：\n###\n    找工作,\n    简历相关,\n    准确,\n    可以考虑,\n    用户负意向,\n    应届生找工作,\n    薪资不合适,\n    位置不合适,\n    换一批,\n    打招呼类闲聊,\n    语气词类闲聊,\n    表示谢意,\n    吐槽类闲聊,\n    学历不合适,\n    工作经验不合适,\n    福利不合适,\n    正向意图,\n    继续推荐,\n    职位不合适,\n    年龄不合适,\n    不找工作了,\n    距离远,\n    高薪资工作,\n    找距离近的工作,\n    不找XXX工作,\n    修改简历\n###\n\n请按照如下要求进行判断：\n1. 请根据对话内容判断用户的意图是否属于以上的意图\n2. 如果属于以上意图则输出对应的意图\n3. 如果不属于以上意图则输出“无意图”\n4. 不要进行解释和说明，直接输出意图\n\n对话内容如下：\n###\n用户：挖掘机\n###\n',
 'input': '',
 'output': '找工作'}

In [69]:
hailong_interview_zhuiwen_46k_ori = read_datas(hailong_interview_zhuiwen_46k_path)
hailong_interview_zhuiwen_46k = []
for elem in hailong_interview_zhuiwen_46k_ori:
    hailong_interview_zhuiwen_46k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
hailong_interview_zhuiwen_46k[0]

{'instruction': '你是一名ai面试官，当用户回答完后，针对用户回答的不明确或者不符合逻辑的回复，需要进行追问。当用户明确的回复了问题时，则输出空字符串。\n其中，input中的question是你的问题，answer是用户的回答。\n要求： \t 1.你输出的追问必须可靠、合规、客观，不允许输出违反法律法规等的内容。\n2.不允许输出其他无关的结果。\n 请根据下面的case示例和要求输出对应的追问:\n#####################\ncase1: \ninput: {"question": "最快什么时候可以入职？", "answer": "没有去上。 你们的要求是什么？"}\noutput: 最快什么时候能够入职呢？\ncase2: \ninput: {"question": "您之前开过哪类型的货车？", "answer": "四米二。"}\noutput: 关于这个问题可以回答平板货车、箱货。请问您之前开过哪种类型的货车呢？\ncase3: \ninput: {"question": "之前有没有干过？干了几年？", "answer": "酒店服务行业是吧？"}\noutput: 方便问下您之前在这个行业干过几年呢？\ncase4: \ninput: {"question": "你是想找长期还是短期，兼职还是全职工作呢？", "answer": "我想找长期的，全职的。"}\noutput: ""\n#####################\n请根据上述case示例和要求完成你的追问，不允许输出其他无关的结果。\n input: \t',
 'input': '{"question": "那您的体重是多少呢？", "answer": "58。1.6m。体重70。"}',
 'output': ''}

In [70]:
hailong_interview_ner_4k_ori = read_datas(hailong_interview_ner_4k_path)
hailong_interview_ner_4k = []
for elem in hailong_interview_ner_4k_ori:
    hailong_interview_ner_4k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
hailong_interview_ner_4k[0]

{'instruction': '你现在要提取面试者的回复的关键信息。请根据input中的question(面试问题)和type(面试岗位类型)，从answer(面试者的回答)中提取对应的关键信息。\n 要求： \t 1.以json格式输出。只输出key(面试问题的类型), value(关键信息的值)这两个字段，格式为：{"key": "", "value": ""}\n2.结果要进行归一化。\n3.提取的结果，如用户的年龄、工作年限等信息要符合常理。如果不符合，对应的value为空字符串。\n4.key只与question(面试问题)有关。\n请根据下面的5个case示例和要求完成input的解析，并输出为json格式：{"key": "", "value": ""}。不允许输出其他无关的结果。\n#####################\ncase1: \ninput: {"question": "请问您之前做过哪类工种?", "type": "普工", "answer": "干过机床工，时间是2年。补充信息: 没有了，下一题"}\noutput: {"key": "工作经验", "value": "机床工"}\ncase2: \ninput: {"question": "之前有没有干过？干了几年？", "type": "普工/操作工", "answer": "没有干过，下一题。"}\noutput: {"key": "工作经验", "value": "无经验"}\ncase3: \ninput: {"question": "您接受哪种工作时间？", "type": "普工/操作工", "answer": "做六休一我可以做，回答完毕。"}\noutput: {"key": "工作时间", "value": "做六休一"}\ncase4: \ninput: {"question": "最快什么时候可以入职？", "type": "运营/客服", "answer": "可能需要一个多月。补充信息: 可能得一个半月吧，大概"}\noutput: {"key": "入职时间", "value": "超过1个月以上入职"}\ncase5: \ninput: {"question": "先做个简单的自我介绍吧（介绍下你的年龄、学历、工作经验）", "type": "服务

In [72]:
yalong_crm_ner_4k_ori = read_datas(yalong_crm_ner_4k_path)
yalong_crm_ner_4k = []
for elem in yalong_crm_ner_4k_ori:
    yalong_crm_ner_4k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
yalong_crm_ner_4k[0]

{'instruction': '解析输入文本的相关槽位信息，包括部门(department)、部门id(department_id)、OA、时间的槽位(format_time)如今日，昨日，本周，本月，近一月，本季等。时间范围的起始时间(start_time)、时间范围的结束时间(end_time)、呼叫时长(dial_time)、知识学习数据槽位1(product)、知识学习数据槽位2(product_v2)、优秀话术数据槽位1(scene)、优秀话术数据槽位2(scene_v2)、优秀话术数据槽位3(scene_type)、城市(city)、商机名(business_opportunity_name)、商机ID(business_opportunity_id)。\n要求：\n 1. format_time中，表述都要归一化在["今日", "昨日", "本周", "上周", "近一周", "近两周", "近三周", "本月", "近一月", "近两月", "上月", "本季"]之内。如不符则为空。\n 2.要求output为json格式，文本中不包含的槽位对应json中的key不输出。\n 3.有效通话时长(dial_time)均转为秒为单位，如15s，30s，60s。\n以下是7个槽位识别的case：\n#####################\ncase1: \ninput: 我想查询【覆盖数据】，部门id为30001，部门或销售为【直销业务部】，销售为【张三】，时间为【今日】。\noutput:{"department_id": "30001", "department": "直销业务部", "OA": "张三", "format_time": "今日"}\ncase2:\ninput: 查询话务数据，部门为ailab，销售为lisi02，时间为2024-04-14~2024-05-07，有效通话时长为3分钟。\noutput:{"department": "ailab", "OA": "lisi02", "start_time": "2024-04-14", "end_time": "2024-05-07", "dial_time": "180s"} \ncase3:\ninput: 我希望学习促消耗中，推单充场景下的的话术处理思路，客户反馈问题已解决

In [73]:
yalong_ka_declarative_sentence_qa_6k_ori = read_datas(yalong_ka_declarative_sentence_qa_6k_path)
yalong_ka_declarative_sentence_qa_6k = []
for elem in yalong_ka_declarative_sentence_qa_6k_ori:
    yalong_ka_declarative_sentence_qa_6k.append({'instruction': elem['instruction'], 'input': elem['input'], 'output': elem['output'].replace('<think>\n\n</think>\n\njson', '')})
yalong_ka_declarative_sentence_qa_6k[0]

{'instruction': '有如下信息：\'\'\'\n{"职位描述": "招聘岗位：居家语聊厅主播无需外出岗位要求：有意向直接投递简历，电话联系详细沟通工作内容：是可以在家工作线上直播的，主要就是在直播厅麦上和进来的游客互动聊天，唱歌、互动游戏、团队配合等等，不露脸，绿色直播！任何觉得不舒服的地方可以提出来，全天直播，可以根据自己的空余时间选择档期，自由直播，轻松直播。每天直播厅内流量推荐，帮新人增加粉丝量，可给新人提供资源，每周会安排培训，进厅后安排师傅带，绿色直播不擦边。账号作品有剪辑负责给制作。工作要求：需要有本人实名认证的快手账号，并且现在没有签约其他公会，每天有固定的直播时间，手机或电脑均可操作，手机加有线耳机就可以工作，直播的时候，保持周边环境音安静，不影响语音直播即可，遵从厅管的指导与工作安排，会细心的一对一辅导，有事提前请假。感觉不合适随时都可以和厅管申请离职。不卡公会。薪资及待遇每天直播三至六小时即可，直播厅全天营业，早中晚都可以，选取时间段是你自己选，最好连续的六个小时，工作期间需要开口说话，（有事提前向厅管请假）工作待遇： 3小时俩月一保，保底4000，提成40%，日结36%月结4%6小时俩月一保，保底6000，提成45%，日结40%，月结5%注：保底为给主持最低保障，实际普遍收入高于保底，本职位不收取任何费用，没有退会费，没有违约金。其他要求:1010674,最低薪资:5000_7000,学历要求:不限,工作年限:不限,最高薪资:7000,角色标签:wenhuachuanmei,薪资水平:5000_7000,起始薪资:5000公司信息：联系人:王女士,公司地址:连云港国际商务大厦,公司名称:哈尔滨润玖文化传媒有限公司,公司规模:1000人以上", "历史会话": "[\'感兴趣\', \'嗯，那这个职位推荐给您考虑一下。您好，我们这边招聘快手语音主播，群播不会冷场，平台流量大，绿色直播，礼物当天可提现，有意向发个简历您点击页面顶部的【发简历】按钮，投下简历null\', \'快手的吖\', \'您先投下简历，我给您详细的介绍下null\', \'有么有别的小平台的\', \'对不起，我们的招聘信息中并没有提到其他小平台的信息。您可以先投下简历，然后在面试过程中向我们的联系人王女士询问这个问题，以获取最准确的信息。null\', \'有

In [74]:
online_datas_457k = []
online_datas_457k.extend(dahao_xinzi_9k)
online_datas_457k.extend(dahao_ershoufang_25k_extra_10k)
online_datas_457k.extend(xudong_b_crf_203k_extra_5w)
online_datas_457k.extend(hangcheng_ka_18k)
online_datas_457k.extend(xurui_ic_new_26_types_20w)
online_datas_457k.extend(tianxiang_ic_support_ans_18k)
online_datas_457k.extend(yuhang_interview_ner_25k)
online_datas_457k.extend(yuhang_agent_crm_ic_5k)
online_datas_457k.extend(feibro_c_ner_31k)
online_datas_457k.extend(feibro_c_ic_29k)
online_datas_457k.extend(hailong_interview_zhuiwen_46k)
online_datas_457k.extend(hailong_interview_ner_4k)
online_datas_457k.extend(yalong_crm_ner_4k)
online_datas_457k.extend(yalong_ka_declarative_sentence_qa_6k)


In [ ]:
# online_datas_457k 与 ner 和 ic 数据合并进行去重

In [75]:
len(online_datas_457k)
online_datas_457k_format = []
for elem in online_datas_457k:
    sample = {
        "messages": [
          {
            "role": "user",
            "content":  elem['instruction'] + elem['input']
          },
          {
            "role": "assistant",
            "content": elem['output']
          }
        ]
    }
    online_datas_457k_format.append(sample)

In [76]:
online_datas_457k_format[10000]

{'messages': [{'role': 'user',
   'content': '### 基于用户历史行为分析租房意向预测poi或小区搜索词，房源信息的具体格式为{区域}{商圈}{小区}{价格}{面积}组成的序列：\n\n### 分析优先级和依据：\n1. 电话咨询行为（最重要）：{五华}{北市区}{和谐世纪}{142万元}{198平方米},{五华}{北市区}{安康园}{127万元}{161平方米}\n2. 浏览点击行为（次重要）：{五华}{北市区}{和谐世纪}{105万元}{148平方米},{五华}{北市区}{远洋风景}{68万元}{87平方米}\n3. 搜索行为（辅助参考）：和谐家园,和谐家园\n   \n### 预测搜索词：'},
  {'role': 'assistant', 'content': '安康园'}]}

In [77]:
save_path = '/code/zhaoxudong03/data_pipelines/training_data/457w/online_datas_457k_format.jsonl'
with open(save_path, 'w', encoding='utf8') as f_save:
    for elem in tqdm(online_datas_457k_format):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 453383/453383 [00:32<00:00, 13860.57it/s]


In [171]:
len(total_datas_no_ace) # 425w
# 将 online_datas 457k 与 通用数据 datas_domain_ner_dep_70w、datas_domain_ic_dep_lst_70w

4254457

In [214]:
total_datas_no_ace_cat_online_datas_format = total_datas_no_ace + online_datas_457k_format
# total_datas_no_ace_cat_online_datas_format = total_datas_no_ace 
total_datas_no_ace_cat_online_datas_format_dep = []
error = 0
for elem in total_datas_no_ace_cat_online_datas_format:
    if elem['messages'][0]['role'] == 'user':
        elem.update({'dep': elem['messages'][0]['content']})
        total_datas_no_ace_cat_online_datas_format_dep.append(elem)

In [215]:
total_datas_no_ace_cat_online_datas_format_dep_df = pd.DataFrame(total_datas_no_ace_cat_online_datas_format_dep)
print(total_datas_no_ace_cat_online_datas_format_dep_df.shape)
total_datas_no_ace_cat_online_datas_format_dep_df_dep = total_datas_no_ace_cat_online_datas_format_dep_df.drop_duplicates(['dep'])
print(total_datas_no_ace_cat_online_datas_format_dep_df_dep.shape)
# 4711702 - 4570534

(4711702, 3)
(4570534, 3)


In [217]:
# drop_domain_datas = pd.DataFrame(online_datas_457k + datas_domain_ic_dep_lst_70w + datas_domain_ner_dep_70w)
# drop_domain_datas = pd.DataFrame(online_datas_457k)
# print(drop_domain_datas.shape)
# drop_domain_datas['dep'] = drop_domain_datas['instruction'] + drop_domain_datas['input']
# drop_domain_datas_dep = drop_domain_datas.drop_duplicates(['dep'])
# print(drop_domain_datas_dep.shape)

total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst = total_datas_no_ace_cat_online_datas_format_dep_df_dep.to_dict(orient='records')

In [225]:

len(total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst), math.isnan(total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst[0]['tools'])

(4570534, True)

In [228]:
total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final = [] 
func = 0; error = 0
for elem in total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst:
    if isinstance(elem['tools'], str):
        total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final.append({'messages': elem['messages'], 'tools': elem['tools']})
        func += 1
    elif math.isnan(elem['tools']):
        total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final.append({'messages': elem['messages']})
    else:
        error += 1        

In [239]:
# total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final[1000]
func
idx = 0
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/general_total_425w_no_ace_reason_cat_online_datas_457w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')
        if 'tools' in elem and idx == 0:
            print(elem)
            idx += 1

 85%|████████▍ | 3883098/4570534 [08:28<04:30, 2542.22it/s] 

{'messages': [{'role': 'user', 'content': "I'm looking to buy a used truck for my construction work, but I want to make sure it's legitimate. The seller provided the VIN: 1FMXK92W8YPA12345 and said it's registered in Georgia. Can you verify: 1. If the VIN is valid and matches a registered vehicle 2. The registration status in Georgia using the license plate number (if available) 3. Any security interests/liens via PPSR? Let me know if you hit any snags—I’m happy to provide more details if needed."}, {'role': 'tool_call', 'content': '{"name": "check_valid_vin", "arguments": {"vin": "1FMXK92W8YPA12345"}}'}, {'role': 'tool_response', 'content': '{"valid": true, "vin_details": {"make": "Ford", "model": "F-150", "year": 2020, "vehicle_type": "Truck", "registration_status": "Active", "registration_state": "GA", "odometer": 82345, "title_status": "Clear", "lienholder": null, "recall_history": "No active recalls"}, "notes": "VIN is valid and registered in Georgia. PPSR lien check complete - no

100%|██████████| 4570534/4570534 [10:14<00:00, 7437.69it/s] 


In [240]:
random.shuffle(total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final)

idx = 0
with open('/code/zhaoxudong03/Train/LLaMA-Factory/data/general_total_425w_no_ace_reason_cat_online_datas_457w_shuffle.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')
        if 'tools' in elem and idx == 0:
            print(elem)
            idx += 1

  0%|          | 357/4570534 [00:00<22:13, 3426.27it/s]

{'messages': [{'role': 'user', 'content': "I'm exploring a new art project that involves repurposing disc golf discs as canvases for abstract painting, inspired by my love for Dalí's surrealism. Could you retrieve a list of discs from the market that are particularly textured or have a matte finish, as those surfaces might best capture the ethereal brushstrokes I envision? Include details like material composition and weight, as I want something durable yet lightweight for my students."}, {'role': 'tool_call', 'content': '{"name": "get_discs", "arguments": {"namelike": "matte", "limit": 20}}'}, {'role': 'tool_response', 'content': '{"results": [{"name": "Matte Eclipse Pro", "manufacturer": "Dynamo Discs", "type": "Mid-Range", "material": "Durable Polyurethane Blend", "weight": "165g", "texture": "Fine Matte Grain", "price": "$18.99"}, {"name": "Surrealist Matte Flow", "manufacturer": "Innova Art Series", "type": "Putter", "material": "Soft TPU Composite", "weight": "170g", "texture": "

100%|██████████| 4570534/4570534 [10:57<00:00, 6950.07it/s]


In [238]:
total_datas_no_ace_cat_online_datas_format_dep_df_dep['tools'].isna().sum()
total_datas_no_ace_cat_online_datas_format_dep_df_dep['tools'].notna().sum()
len(total_datas_no_ace_cat_online_datas_format_dep_df_dep_lst_final)

4570534

In [162]:
len(dahao_xinzi_9k), len(dahao_ershoufang_25k_extra_10k), len(xudong_b_crf_203k_extra_5w), len(hangcheng_ka_18k), len(xurui_ic_new_26_types_20w), len(tianxiang_ic_support_ans_18k), len(yuhang_interview_ner_25k), len(yuhang_agent_crm_ic_5k), len(feibro_c_ner_31k), len(feibro_c_ic_29k), len(hailong_interview_zhuiwen_46k), len(hailong_interview_ner_4k), len(yalong_crm_ner_4k), len(yalong_ka_declarative_sentence_qa_6k) 

(9000,
 10000,
 50000,
 18188,
 200000,
 18000,
 24870,
 4752,
 31465,
 29516,
 46537,
 4441,
 4238,
 6250)

In [2]:
# 改写的livebench 两个子项，数据分析、语言
import os 
import json 
rw_livebench = '/code/zhaoxudong03/Train/LLaMA-Factory/data/zxd/datas/livebench_rewrite_Qwen3_2507/'
rw_livebnech_datas = []
for file_name in os.listdir(rw_livebench):
    # if file_name in ['rewrite_datas_2507_gen_ans_data_anaysis.json', 'rewrite_datas_2507_gen_ans_language.json']:
    if file_name.endswith('json'):
        file_path = os.path.join(rw_livebench, file_name)
        with open(file_path, 'r', encoding='utf8') as f:
            for line in f.readlines():
                cur_ = json.loads(line)
                sample = {'instruction': cur_['instruction'], 'input': '', 'output': cur_['output']}
                rw_livebnech_datas.append(sample)
        print(len(rw_livebnech_datas))

11517
26307
55897
73424
108217
125865


In [260]:
len(datas_202w), datas_202w[0]
df_202w = pd.DataFrame(datas_202w)
df_202w.head()

,instruction,input,output
0,Your answer must contain exactly 3 bullet poin...,,* This is point 1. \n* Explain the process of...
1,Your answer must contain exactly 3 bullet poin...,,* This is point 1: To improve your public spea...
2,Your response should contain less than 5 sente...,,Section 1 \nBy paying in 10 days instead of 3...
3,Your answer must contain exactly 3 bullet poin...,,* Locational arbitrage can be executed by buyi...
4,What are the main challenges of implementing a...,,<<Overcoming Hurdles in Business Machine Learn...


In [261]:
df_202w.shape

(2020642, 3)

In [265]:
# 用来提取 202w 数据的 data_source 并根据这个字段 获取这份数据的embedding 
path_cls_ori_1 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/totals_datas_cls.jsonl' # 870000
path_cls_ori_2 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/totals_datas_cls_2.jsonl' # 4178069

In [264]:
datas_cls_1 = read_datas(path_cls_ori_1)

In [266]:
datas_cls_2 = read_datas(path_cls_ori_2)

In [267]:
datas_cls_cat = datas_cls_2 + datas_cls_1
datas_cls_df = pd.DataFrame(datas_cls_cat)

In [268]:
datas_cls_df.head(), datas_cls_df.shape

(       id                                      source  \
 0  870133  AM-DeepSeek-R1-0528-Distilled-other-177243   
 1  870176  AM-DeepSeek-R1-0528-Distilled-other-177289   
 2  870302  AM-DeepSeek-R1-0528-Distilled-other-177422   
 3  870244  AM-DeepSeek-R1-0528-Distilled-other-177362   
 4  870768  AM-DeepSeek-R1-0528-Distilled-other-177936   
 
                                               prompt  \
 0  You are an AI assistant. User will you give yo...   
 1  Question: What entity did Bronck get his land ...   
 2  Edit the following sentence, removing the repe...   
 3  You are an AI assistant. User will you give yo...   
 4  What's the exact number of hours in a non-leap...   
 
                                               output finish_reason  \
 0  ```json\n{\n  "Clarity": {\n    "Score": 10,\n...          stop   
 1  ```json\n{\n  "Clarity": {\n    "Score": 10,\n...          stop   
 2  ```json\n{\n  "Clarity": {\n    "Score": 10,\n...          stop   
 3  ```json\n{\n  "Cla

In [269]:
# 202w 中包含其他的类似 zh_110k if 等其他内容
pd_merge = pd.merge(datas_cls_df, df_202w, left_on=['prompt'], right_on=['instruction'], how='right')

In [272]:
pd_merge.head(), pd_merge.shape

(         id                                  source  \
 0  223122.0   AM-DeepSeek-R1-0528-Distilled-if-7648   
 1  248911.0  AM-DeepSeek-R1-0528-Distilled-if-38160   
 2  243954.0  AM-DeepSeek-R1-0528-Distilled-if-32025   
 3  257168.0  AM-DeepSeek-R1-0528-Distilled-if-48469   
 4  241543.0  AM-DeepSeek-R1-0528-Distilled-if-29047   
 
                                               prompt  \
 0  Your answer must contain exactly 3 bullet poin...   
 1  Your answer must contain exactly 3 bullet poin...   
 2  Your response should contain less than 5 sente...   
 3  Your answer must contain exactly 3 bullet poin...   
 4  What are the main challenges of implementing a...   
 
                                             output_x finish_reason  \
 0  ```json\n{\n  "Clarity": {\n    "Score": 9,\n ...          stop   
 1  ```json\n{\n  "Clarity": {\n    "Score": 10,\n...          stop   
 2  ```json\n{\n  "Clarity": {\n    "Score": 9,\n ...          stop   
 3  ```json\n{\n  "Clarity": {\n  

In [273]:
pd_merge_inner = pd.merge(datas_cls_df, df_202w, left_on=['prompt'], right_on=['instruction'], how='inner')


In [276]:
pd_merge_inner_dep = pd_merge_inner.drop_duplicates(['instruction'])

In [278]:
pd_merge_inner_dep.head()

,id,source,prompt,output_x,finish_reason,output_tokens,instruction,input,output_y
0,870702,AM-DeepSeek-R1-0528-Distilled-other-177864,"Premise: ""A male and female are holding hands ...","```json\n{\n ""Clarity"": {\n ""Score"": 10,\n...",stop,262,"Premise: ""A male and female are holding hands ...",,The premise states that the male and female ar...
1,870136,AM-DeepSeek-R1-0528-Distilled-other-177246,Find the remainder when 2 raised to the power ...,"```json\n{\n ""Clarity"": {\n ""Score"": 10,\n...",stop,280,Find the remainder when 2 raised to the power ...,,We want to find the **remainder when $ 2^{17} ...
2,870261,AM-DeepSeek-R1-0528-Distilled-other-177379,"[QUESTION] Premise: ""A young boy excitedly sho...","```json\n{\n ""Clarity"": {\n ""Score"": 10,\n...",stop,283,"[QUESTION] Premise: ""A young boy excitedly sho...",,"The phrase ""Guy instructing people in back of ..."
3,870226,AM-DeepSeek-R1-0528-Distilled-other-177343,Find the logical fallacy in the statement.\nIf...,"```json\n{\n ""Clarity"": {\n ""Score"": 9,\n ...",stop,290,Find the logical fallacy in the statement.\nIf...,,"The logical fallacy in the statement **""If you..."
4,870710,AM-DeepSeek-R1-0528-Distilled-other-177872,You are an AI assistant that helps people find...,"```json\n{\n ""Clarity"": {\n ""Score"": 9,\n ...",stop,292,You are an AI assistant that helps people find...,,Let's think step-by-step:\n\n1. The premise st...


In [280]:
need_embed = pd_merge_inner_dep['source'].values.tolist()

In [279]:
# 一行一行读取 embedding 如果 embedding 内容在 pd_merge_inner_dep 的 source 中，保留 embedding 
def yield_data1(path):
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            yield line 

In [282]:
path_embed_1 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/strict_dep_embeded_0-6B.jsonl'
path_embed_2 = '/code/zhaoxudong03/data_pipelines/training_data_zxd/strict_dep_embeded_0-6B-2.jsonl'


In [284]:
datas_202w_embed = []
nums_ = 0
for elem in tqdm(yield_data1(path_embed_1)):
    cur_ = json.loads(elem)
    if cur_['source'] in need_embed:
        datas_202w_embed.append(cur_)
        nums_ += 1

91537it [53:22, 28.58it/s]


KeyboardInterrupt: 